In [1]:
# # Cell 0 — Dataset Setup & Format Conversion
# # ============================================================
# # YOUR FOLDER STRUCTURE (from VS Code Explorer):
# #
# #   T2520716/
# #   ├── CHARADES_STA_DATASET/
# #   │   ├── annotations/annotations/annotations/
# #   │   │   ├── charades_sta_train.txt   ← raw annotation file
# #   │   │   ├── charades_sta_test.txt    ← raw annotation file
# #   │   │   └── charades_subset.json
# #   │   ├── videos/videos/videos/        ← your .mp4 files live here
# #   │   ├── train.csv
# #   │   └── test.csv
# #   ├── TEMPORAL_GROUNDING_VSCODE.ipynb  ← this notebook
# #   └── VLMI.ipynb
# #
# # WHAT THIS CELL DOES:
# #   1. Reads your .txt annotation files (CharadesSTA native format)
# #   2. Reads video durations from train.csv / test.csv
# #   3. Converts everything to the .json format the notebook expects
# #   4. Writes train.json, val.json, test.json into ./CharadesSTA/Annotations/
# #   5. Creates a symlink/copy so the Config paths resolve correctly
# #
# # RUN THIS CELL ONCE before Cell 3 (Config). You do not need to re-run
# # it on subsequent training sessions — the .json files persist on disk.
# # ============================================================

# import os, json, csv, shutil
# from pathlib import Path
# from collections import defaultdict

# # ── Step 1: Point to your actual paths ───────────────────────────────────────
# # These match exactly what is visible in your VS Code Explorer.
# BASE          = Path(".")   # notebook is in T2520716/

# ANNO_DIR      = BASE / "CHARADES_STA_DATASET" / "annotations" / "annotations" / "annotations"
# VIDEO_DIR_SRC = BASE / "CHARADES_STA_DATASET" / "videos" / "videos" / "videos"

# TRAIN_TXT     = ANNO_DIR / "charades_sta_train.txt"
# TEST_TXT      = ANNO_DIR / "charades_sta_test.txt"
# TRAIN_CSV     = BASE / "CHARADES_STA_DATASET" / "train.csv"
# TEST_CSV      = BASE / "CHARADES_STA_DATASET" / "test.csv"

# # ── Step 2: Output paths (what the notebook Config expects) ───────────────────
# OUT_ROOT      = BASE / "CHARADES_STA_DATASET"
# OUT_ANNO      = OUT_ROOT / "Annotations"
# OUT_VIDEO     = OUT_ROOT / "Videos"

# OUT_ROOT.mkdir(parents=True, exist_ok=True)
# OUT_ANNO.mkdir(parents=True, exist_ok=True)

# # ── Step 3: Symlink video folder (avoids copying ~50GB of video files) ────────
# # Creates CharadesSTA/Videos → CHARADES_STA_DATASET/videos/videos/videos
# if not OUT_VIDEO.exists():
#     try:
#         OUT_VIDEO.symlink_to(VIDEO_DIR_SRC.resolve())
#         print(f"[OK] Video symlink created: {OUT_VIDEO} → {VIDEO_DIR_SRC.resolve()}")
#     except Exception as e:
#         # Fallback on Windows (symlinks need admin rights) — use junction or warn
#         print(f"[WARN] Symlink failed ({e})")
#         print(f"       On Windows: run VS Code as Administrator, or manually copy")
#         print(f"       {VIDEO_DIR_SRC}  →  {OUT_VIDEO}")
# else:
#     print(f"[OK] Video path exists: {OUT_VIDEO}")

# # ── Step 4: Load durations from CSV ──────────────────────────────────────────
# # train.csv and test.csv contain video metadata including duration.
# # Standard CharadesSTA CSV columns: id, subject, scene, quality, relevance,
# #   verified, script, objects, descriptions, action, length (duration in seconds)

# def load_durations(csv_path):
#     durations = {}
#     if not csv_path.exists():
#         print(f"[WARN] {csv_path} not found — duration will default to 0.0")
#         return durations
#     with open(csv_path, newline="", encoding="utf-8") as f:
#         reader = csv.DictReader(f)
#         for row in reader:
#             vid_id = row.get("id", "").strip()
#             # Duration column is named "length" in CharadesSTA CSV
#             length = row.get("length", row.get("duration", "0")).strip()
#             try:
#                 durations[vid_id] = float(length)
#             except ValueError:
#                 durations[vid_id] = 0.0
#     print(f"[OK] Loaded {len(durations)} video durations from {csv_path.name}")
#     return durations

# train_durations = load_durations(TRAIN_CSV)
# test_durations  = load_durations(TEST_CSV)
# all_durations   = {**train_durations, **test_durations}

# # ── Step 5: Parse .txt annotation files ──────────────────────────────────────
# # Format of each line:
# #   <video_id> <start_sec> <end_sec>##<natural language sentence>
# # Example:
# #   3MSZA 0.0 4.9##a person is putting a book on a bookshelf.

# def parse_txt(txt_path, durations):
#     """
#     Parse a CharadesSTA .txt file and return a list of sample dicts
#     in the JSON format expected by the CharadesSTA Dataset class:

#     [
#       {
#         "video_id":    "3MSZA.mp4",
#         "duration":    30.2,
#         "annotations": [
#           {"sentence": "a person puts a book...", "timestamp": [0.0, 4.9]}
#         ]
#       },
#       ...
#     ]
#     """
#     if not txt_path.exists():
#         print(f"[ERROR] Not found: {txt_path}")
#         return []

#     grouped = defaultdict(list)   # video_id → list of (sentence, start, end)

#     with open(txt_path, "r", encoding="utf-8") as f:
#         for line_no, line in enumerate(f, 1):
#             line = line.strip()
#             if not line:
#                 continue
#             parts = line.split()
#             if len(parts) < 4:
#                 print(f"[WARN] Line {line_no} malformed: {line[:60]}")
#                 continue
#             vid_id   = parts[0]
#             try:
#                 start    = float(parts[1])
#                 end      = float(parts[2])
#             except ValueError:
#                 print(f"[WARN] Line {line_no} bad timestamps: {line[:60]}")
#                 continue
#             sentence = " ".join(parts[3:])
#             grouped[vid_id].append((sentence.strip(), start, end))

#     samples = []
#     for vid_id, annos in grouped.items():
#         # Video files are .mp4 in CharadesSTA
#         video_filename = f"{vid_id}.mp4"
#         duration = durations.get(vid_id, 0.0)
#         if duration == 0.0:
#             # Estimate from max end timestamp as fallback
#             duration = max(e for _, _, e in annos) + 1.0

#         samples.append({
#             "video_id":    video_filename,
#             "duration":    duration,
#             "annotations": [
#                 {"sentence": sent, "timestamp": [start, end]}
#                 for sent, start, end in annos
#             ]
#         })

#     print(f"[OK] Parsed {len(samples)} videos, "
#           f"{sum(len(s['annotations']) for s in samples)} annotations "
#           f"from {txt_path.name}")
#     return samples

# train_data = parse_txt(TRAIN_TXT, all_durations)
# test_data  = parse_txt(TEST_TXT,  all_durations)

# # ── Step 6: Split train → train (90%) + val (10%) ────────────────────────────
# import random
# random.seed(42)
# random.shuffle(train_data)
# split_idx  = int(len(train_data) * 0.9)
# val_data   = train_data[split_idx:]
# train_data = train_data[:split_idx]

# print(f"[OK] Split → train: {len(train_data)} | val: {len(val_data)} | test: {len(test_data)}")

# # ── Step 7: Write JSON files ──────────────────────────────────────────────────
# for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
#     out_path = OUT_ANNO / f"{name}.json"
#     with open(out_path, "w", encoding="utf-8") as f:
#         json.dump(data, f, indent=2)
#     print(f"[OK] Written: {out_path}  ({len(data)} entries)")

# # ── Step 8: Sanity check ──────────────────────────────────────────────────────
# print()
# print("=== Sample entry from train.json ===")
# if train_data:
#     print(json.dumps(train_data[0], indent=2))

# print()
# print("=== Path check ===")
# for p in [OUT_ANNO / "train.json", OUT_ANNO / "val.json", OUT_ANNO / "test.json", OUT_VIDEO]:
#     status = "✓ exists" if p.exists() else "✗ MISSING"
#     print(f"  {status}  {p}")

# print()
# print("[DONE] Dataset setup complete. You can now run Cell 1 onwards.")


## Cell 1 — Install & Import Dependencies

In [2]:
# Cell 1: Environment Setup — VS Code + i7-14700K / RTX 4080 Super 16GB / 64GB DDR5
# ============================================================
# VS Code adaptation: install packages ONCE via terminal before running cells.
#
# In your VS Code terminal (activate your conda/venv first), run:
#
#   pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
#   pip install transformers==4.40.2 timm==0.9.16 decord==0.6.0 einops==0.7.0
#   pip install opencv-python==4.9.0.80 tensorboard==2.16.2
#   pip install tqdm matplotlib seaborn pandas numpy scipy
#
# cu124 (CUDA 12.4) is recommended for the RTX 4080 Super (Ada Lovelace, sm_89):
#   - Enables Flash Attention v2 dispatch
#   - Improved BF16/TF32 throughput on Ada tensor cores
#   - cu121 also works but misses the 12.4 kernel optimisations
#
# Verify GPU visibility:
#   python -c "import torch; print(torch.cuda.get_device_name(0))"
# ============================================================

import sys

def _check(pkg):
    import importlib
    mod = pkg.split("==")[0].replace("-", "_")
    try:
        importlib.import_module(mod)
        return True
    except ImportError:
        print(f"[WARN] '{pkg}' not installed. Run: pip install {pkg}")
        return False

required = ["torch", "transformers", "timm", "decord", "einops", "cv2", "tensorboard", "tqdm"]
ok = all(_check(p) for p in required)

if ok:
    import torch
    if not torch.cuda.is_available():
        print("[ERROR] CUDA not available — check drivers and PyTorch cu124 install.")
    else:
        print(f"[OK]     Environment ready")
        print(f"[GPU]    {torch.cuda.get_device_name(0)}")
        print(f"[VRAM]   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        print(f"[CUDA]   {torch.version.cuda}")
        print(f"[Torch]  {torch.__version__}")


[WARN] 'transformers' not installed. Run: pip install transformers


In [3]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "opencv-python", "numpy"],
    capture_output=True, text=True
)
print(result.stdout)
print("Python:", sys.version)

Name: opencv-python
Version: 4.9.0.80
Summary: Wrapper package for OpenCV python bindings.
Home-page: https://github.com/opencv/opencv-python
Author: 
Author-email: 
License: Apache 2.0
Location: c:\VLMI\.venv\Lib\site-packages
Requires: numpy
Required-by: 
---
Name: numpy
Version: 2.4.6
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License-Expression: BSD-3-Clause AND 0BSD AND MIT AND Zlib AND CC0-1.0
Location: c:\VLMI\.venv\Lib\site-packages
Requires: 
Required-by: contourpy, decord, matplotlib, opencv-python, pandas, scipy, seaborn, tensorboard, torchvision

Python: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


## Cell 2 — Core Imports & Reproducibility

In [4]:
# Cell 2: Core Imports & Reproducibility
# ============================================================
import os, json, math, random, warnings, logging
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any
from functools import partial
from itertools import chain
from collections import defaultdict

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# VS Code: torch.cuda.amp is deprecated in PyTorch >= 2.3; use torch.amp.
# Both aliases imported for backward compatibility.
try:
    from torch.amp import autocast, GradScaler
except ImportError:
    from torch.cuda.amp import autocast, GradScaler  # fallback for PyTorch < 2.3
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import torchvision.transforms as T

from transformers import BertTokenizer, BertModel
from timm.models.layers import DropPath, trunc_normal_
from einops import rearrange, repeat
from scipy.optimize import linear_sum_assignment   # ← add this
import time as _time_dt                            # ← add this (used in Cell 21b)

warnings.filterwarnings("ignore")

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    # On CUDA: benchmark=True lets cuDNN pick the fastest kernel (critical for
    # 3D convolutions in PatchEmbed3D and window attention).
    # deterministic=True + AMP on A100/H100 kills ~30% throughput — we skip it
    # during training and only enable it in dedicated eval/debug runs.
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark     = True
        torch.backends.cudnn.deterministic = False
    else:
        torch.backends.cudnn.benchmark     = False
        torch.backends.cudnn.deterministic = True
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    # ── RTX 4080 Super (Ada Lovelace, sm_89) hardware optimisations ──────────
    # TF32: supported on Ampere+ (sm_80+); ~10x faster than FP32 matmul with
    # negligible numerical difference. Safe for both training and inference.
    torch.backends.cuda.matmul.allow_tf32  = True   # dense matmuls (Linear layers)
    torch.backends.cudnn.allow_tf32        = True   # 3D convolutions (patch embedd3D)

    # benchmark stays True (set in set_seed above); reaffirmed here for clarity.
    torch.backends.cudnn.benchmark         = True
    torch.backends.cudnn.deterministic     = False

    print(f"[GPU]    {torch.cuda.get_device_name(0)}")
    print(f"[VRAM]   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"[CUDA]   {torch.version.cuda} | TF32 ON | cuDNN benchmark ON")
    print(f"[CPU]    {os.cpu_count()} logical cores (i7-14700K: 20C / 28T)")
print(f"[Device] {DEVICE}")

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)
logger.info("[v2] Imports complete.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\VLMI\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\VLMI\.venv\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "c:\VLMI\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "c:\VLMI\.venv\Lib\site-packages\tornado\platform\asyncio.py", line 211,

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

# Cell 2b: Checkpoint reset utility


In [ ]:
# Cell 2b: Checkpoint reset utility
# ============================================================
# Run this cell when changing NUM_EPOCHS or LR config to clear
# stale scheduler state.
# ============================================================

import shutil
from pathlib import Path

RESET_VIT  = False

for bname, flag in [("ViT", RESET_VIT)]:
    ckpt_dir = Path("./checkpoints_v2") / bname
    latest   = ckpt_dir / "latest.pt"
    if flag:
        if ckpt_dir.exists():
            shutil.rmtree(ckpt_dir)
            print(f"[RESET] Deleted {ckpt_dir}")
        else:
            print(f"[--] {bname} checkpoint dir not found, nothing to delete")
    else:
        status = "exists" if latest.exists() else "not found"
        print(f"[SKIP] {bname} reset skipped (latest.pt {status})")


[SKIP] ViT reset skipped (latest.pt not found)


## Cell 3 — Global Configuration (v2 — updated dims)

In [ ]:
# Cell 3: Global Configuration — ViT-only
# ============================================================
# VRAM analysis (RTX 4080 Super 16 GB, BF16, Adam optimizer):
#
#   Fixed costs  (weights + Adam states + CUDA overhead): ~1.8 GB
#   Activations  (16 frames, batch 8):                   ~5.12 GB
#   ── Total estimated VRAM:                              ~6.9 GB
#   ── Headroom:                                          ~9.1 GB ✓ very safe
#
# Effective batch: 8 × GRAD_ACCUM(2) = 16
# ============================================================

class Config:
    # ── Paths ────────────────────────────────────────────────
    DATA_ROOT            = Path("./CHARADES_STA_DATASET")
    VIDEO_DIR            = DATA_ROOT / "Videos"
    ANNO_DIR             = DATA_ROOT / "Annotations"
    TRAIN_JSON           = ANNO_DIR  / "train.json"
    VAL_JSON             = ANNO_DIR  / "val.json"
    TEST_JSON            = ANNO_DIR  / "test.json"
    CKPT_DIR             = Path("./checkpoints_v2")
    LOG_DIR              = Path("./logs_v2")

    # ── Video ViT ────────────────────────────────────────────
    VIT_PATCH_T    = 2
    VIT_PATCH_H    = 32
    VIT_PATCH_W    = 32
    VIT_EMBED_DIM  = 768
    VIT_DEPTH      = 6
    VIT_NUM_HEADS  = 12
    VIT_MLP_RATIO  = 4.0
    VIT_DROP       = 0.0
    VIT_ATTN_DROP  = 0.0
    VIT_DROP_PATH  = 0.2
    PRETRAINED_VIT_PATH = None

    # ── Video Input ──────────────────────────────────────────
    NUM_FRAMES     = 16
    FRAME_H        = 224
    FRAME_W        = 224
    IN_CHANNELS    = 3

    # ── VIDEO_FEAT_DIM: must match VIT_EMBED_DIM ─────────────
    VIDEO_FEAT_DIM = 768   # same as VIT_EMBED_DIM

    # ── BERT ─────────────────────────────────────────────────
    BERT_MODEL          = "bert-base-uncased"
    MAX_TEXT_LEN        = 32
    TEXT_FEAT_DIM       = 768
    BERT_FREEZE_LAYERS  = 6

    # ── Projection ───────────────────────────────────────────
    PROJ_MID_DIM   = 512
    SHARED_DIM     = 512

    # ── Fusion ───────────────────────────────────────────────
    FUSION_HEADS      = 8
    FUSION_FFN_DIM    = 1024
    NUM_FUSION_LAYERS = 4
    FUSION_DROP       = 0.2

    # ── Span head ────────────────────────────────────────────
    NUM_QUERIES     = 5
    CONF_THRESHOLD  = 0.5
    MLP_HIDDEN_DIMS = [256, 128]
    MLP_DROP        = 0.2

    # ── Training ─────────────────────────────────────────────
    BATCH_SIZE    = 8
    GRAD_ACCUM    = 2
    NUM_EPOCHS    = 30
    TOTAL_EPOCHS  = 30

    LR_BERT       = 3e-5
    LR_HEAD       = 1e-5

    WEIGHT_DECAY  = 0.05
    WARMUP_EPOCHS = 8
    CLIP_GRAD     = 0.1
    AMP           = True
    AMP_DTYPE     = torch.bfloat16

    # Windows-safe DataLoader settings
    NUM_WORKERS        = 0
    PIN_MEMORY         = False
    PERSISTENT_WORKERS = False
    PREFETCH_FACTOR    = None

    # ── Evaluation ───────────────────────────────────────────
    EVAL_METRICS_IOU = [0.3, 0.5, 0.7]

cfg = Config()
cfg.CKPT_DIR.mkdir(parents=True, exist_ok=True)
cfg.LOG_DIR.mkdir(parents=True, exist_ok=True)
cfg.VIDEO_DIR = Path("./CHARADES_STA_DATASET/videos/videos/videos")

# ── Early stopping ────────────────────────────────────────────────────────────
cfg.ES_PATIENCE  = 7
cfg.ES_MIN_DELTA = 0.2
cfg.ES_MODE      = 'max'

logger.info(f"[ViT-only] Config ready. NUM_FRAMES={cfg.NUM_FRAMES}, VIT_DEPTH={cfg.VIT_DEPTH}, "
            f"FUSION_LAYERS={cfg.NUM_FUSION_LAYERS}, BERT_FREEZE={cfg.BERT_FREEZE_LAYERS}")
logger.info(f"[ViT-only] LR — BERT: {cfg.LR_BERT}, Head: {cfg.LR_HEAD}")
logger.info(f"[ViT-only] Effective batch = {cfg.BATCH_SIZE * cfg.GRAD_ACCUM} "
            f"| Est. VRAM ≈ 6.9 GB | Headroom ≈ 9.1 GB")
logger.info(f"[ViT-only] Regularisation — VIT_DROP_PATH={cfg.VIT_DROP_PATH}, "
            f"WEIGHT_DECAY={cfg.WEIGHT_DECAY}, FUSION_DROP={cfg.FUSION_DROP}")


NameError: name 'torch' is not defined

## Cell 4 — Mlp Helper (used by ViTBlock)


In [ ]:
# Cell 4: Mlp helper class — used by ViTBlock in Cell 8b
# ============================================================

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.0):
        super().__init__()
        h = hidden_features or in_features; o = out_features or in_features
        self.fc1 = nn.Linear(in_features, h); self.act = nn.GELU()
        self.fc2 = nn.Linear(h, o); self.drop = nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))

logger.info("Mlp defined ✓")


## Cell 5 — VideoViT: Video Vision Transformer Backbone


In [ ]:
# Cell 5: VideoViT -- Video Vision Transformer Backbone
# ============================================================
# Architecture:
#   * 3-D Conv3d patch embedding: kernel (PT=2, PH=16, PW=16)
#     -> for 16-frame 224x224: 8 x 7 x 7 = 392 patch tokens
#   * Learnable [CLS] token + learnable 1-D positional embeddings
#   * 12 standard bidirectional transformer blocks (Pre-LN, no causal mask)
#   * Output: (B, Lv, embed_dim)  Lv = (T/PT)*(H/PH)*(W/PW)
#
#   Both output VIDEO_FEAT_DIM=768 --> projection, fusion, span head unchanged.
# ============================================================


class ViTBlock(nn.Module):
    """
    Standard ViT transformer block with Pre-LN (Vaswani 2017 variant):
        x <- x + DropPath(SelfAttn(LN(x)))
        x <- x + DropPath(FFN(LN(x)))
    Full bidirectional attention (no causal mask -- correct for video encoding).
    Reuses Cell 6 Mlp class.
    """
    def __init__(self, dim: int, num_heads: int, mlp_ratio: float = 4.0,
                 drop: float = 0.0, attn_drop: float = 0.0, drop_path: float = 0.0):
        super().__init__()
        self.norm1     = nn.LayerNorm(dim)
        self.attn      = nn.MultiheadAttention(dim, num_heads, dropout=attn_drop, batch_first=True)
        self.norm2     = nn.LayerNorm(dim)
        self.mlp       = Mlp(dim, int(dim * mlp_ratio), drop=drop)   # reuses Cell 6 Mlp
        self.drop_path = DropPath(drop_path) if drop_path > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_n = self.norm1(x)
        attn_out, _ = self.attn(x_n, x_n, x_n)
        x = x + self.drop_path(attn_out)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class VideoViT(nn.Module):
    """
    Video Vision Transformer (ViT-B/16 style) with 3-D patch embedding.

    Input  : (B, C, T, H, W)
    Output : (B, Lv, embed_dim)   Lv = (T/PT) * (H/PH) * (W/PW)
             Temporal structure preserved -- no windowing, no hierarchical pooling.

    Complexity: O(Lv^2) FLOPs (global attention), richer temporal detail.
    """

    def __init__(
        self,
        in_chans:       int   = 3,
        num_frames:     int   = 8,
        img_h:          int   = 224,
        img_w:          int   = 224,
        patch_t:        int   = 2,
        patch_h:        int   = 16,
        patch_w:        int   = 16,
        embed_dim:      int   = 768,
        depth:          int   = 12,
        num_heads:      int   = 12,
        mlp_ratio:      float = 4.0,
        drop_rate:      float = 0.0,
        attn_drop:      float = 0.0,
        drop_path_rate: float = 0.1,
    ):
        super().__init__()
        self.embed_dim = embed_dim

        # 3-D patch embedding via Conv3d
        self.patch_embed = nn.Conv3d(
            in_chans, embed_dim,
            kernel_size=(patch_t, patch_h, patch_w),
            stride     =(patch_t, patch_h, patch_w),
        )
        T_tok = num_frames // patch_t
        H_tok = img_h     // patch_h
        W_tok = img_w     // patch_w
        num_patches = T_tok * H_tok * W_tok   # default: 4 * 14 * 14 = 784

        # Learnable [CLS] token and positional embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop  = nn.Dropout(drop_rate)

        # Transformer blocks with stochastic depth schedule
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            ViTBlock(embed_dim, num_heads, mlp_ratio, drop_rate, attn_drop, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        self._init_weights()

    def _init_weights(self):
        trunc_normal_(self.cls_token, std=0.02)
        trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T, H, W)
        B = x.shape[0]

        # 3-D patch embed: (B, C, T, H, W) -> (B, D, T', H', W') -> (B, Lv, D)
        x = self.patch_embed(x).flatten(2).transpose(1, 2)

        # Prepend CLS token and add learnable position embedding
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)          # (B, 1+Lv, D)
        x   = self.pos_drop(x + self.pos_embed)

        # Forward through all transformer blocks
        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)

        # Return patch tokens only (drop CLS).
        # Output contract: (B, Lv, embed_dim). Temporal structure preserved.
        return x[:, 1:, :]   # (B, Lv, embed_dim)


def build_video_vit(cfg) -> VideoViT:
    return VideoViT(
        in_chans       = cfg.IN_CHANNELS,
        num_frames     = cfg.NUM_FRAMES,
        img_h          = cfg.FRAME_H,
        img_w          = cfg.FRAME_W,
        patch_t        = cfg.VIT_PATCH_T,
        patch_h        = cfg.VIT_PATCH_H,
        patch_w        = cfg.VIT_PATCH_W,
        embed_dim      = cfg.VIT_EMBED_DIM,
        depth          = cfg.VIT_DEPTH,
        num_heads      = cfg.VIT_NUM_HEADS,
        mlp_ratio      = cfg.VIT_MLP_RATIO,
        drop_rate      = cfg.VIT_DROP,
        attn_drop      = cfg.VIT_ATTN_DROP,
        drop_path_rate = cfg.VIT_DROP_PATH,
    )


_vit = build_video_vit(cfg)
_vit_tokens = (cfg.NUM_FRAMES // cfg.VIT_PATCH_T
               * cfg.FRAME_H // cfg.VIT_PATCH_H
               * cfg.FRAME_W // cfg.VIT_PATCH_W)
logger.info(f"VideoViT: {sum(p.numel() for p in _vit.parameters())/1e6:.1f} M params | {_vit_tokens} tokens/sample")
del _vit
logger.info("VideoViT + build_video_vit() defined [OK]")


## Cell 9 — BERT Text Encoder (fully trainable — unchanged)

In [ ]:
# Cell 9 [v3 FIX — BUG 3]: BERT Text Encoder with selective layer freezing
# ============================================================
# BUG 3 FIX — BERT fully trainable on ~10K samples:
#   v2: ALL 110M BERT parameters trainable on CharadesSTA
#       (~12K training samples) → severe overfitting.
#       Standard practice for small datasets is to freeze
#       the bottom 6-8 of 12 BERT layers, keeping only the
#       top layers and the pooler fully trainable.
#
#   v3 FIX: freeze_layers=6 (bottom half frozen):
#       - Embedding layer     → frozen
#       - Encoder layers 0-5  → frozen  (low-level syntax/morphology)
#       - Encoder layers 6-11 → trainable (high-level semantics)
#       - Pooler              → trainable
#   This halves trainable BERT params (~55M → ~28M) while
#   preserving high-level grounding-relevant representations.
# ============================================================

class TextEncoder(nn.Module):
    """
    BERT text encoder with selective layer freezing.

    Args:
        model_name    : HuggingFace model identifier
        freeze_layers : Number of bottom BERT encoder layers to freeze.
                        0  = all trainable (v2 behaviour — overfits on small data)
                        6  = freeze bottom half (recommended for CharadesSTA)
                        12 = freeze all encoder layers (only pooler trains)
    """

    def __init__(self, model_name: str = 'bert-base-uncased', freeze_layers: int = 6):
        super().__init__()
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.bert      = BertModel.from_pretrained(model_name)

        # ── Selective freezing ────────────────────────────────
        if freeze_layers > 0:
            # Freeze the embedding layer entirely
            for p in self.bert.embeddings.parameters():
                p.requires_grad = False

            # Freeze the bottom `freeze_layers` transformer layers
            for i, layer in enumerate(self.bert.encoder.layer):
                if i < freeze_layers:
                    for p in layer.parameters():
                        p.requires_grad = False

        # Pooler always remains trainable (task-specific)
        for p in self.bert.pooler.parameters():
            p.requires_grad = True

        frozen    = sum(p.numel() for p in self.bert.parameters() if not p.requires_grad)
        trainable = sum(p.numel() for p in self.bert.parameters() if p.requires_grad)
        logger.info(f'[TextEncoder] Frozen: {frozen/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M '
                    f'(freeze_layers={freeze_layers})')

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True,
        )
        return out.last_hidden_state, out.pooler_output  # (B,L,768), (B,768)


logger.info('TextEncoder (BERT, freeze_layers=cfg.BERT_FREEZE_LAYERS, BUG 3 fixed) defined ✓')


## Cell 10 — FIX 5: 3-Stage Residual Projection

In [ ]:
# Cell 10: ResidualProjection — 3-Stage Bottleneck (FIX 5)
# ============================================================
# v1 PROBLEM: Single 2-layer MLP: 768 → 256 → 256
#   Compressing 768 dims to 256 in one linear step destroys
#   ~67% of the feature space abruptly. BERT and Swin spend
#   their full capacity building 768-dim representations;
#   collapsing them this hard before fusion limits recall.
#
# v2 FIX: 3-stage residual projection with linear skip:
#   768 → 640 (LN + GELU) → 512 (LN + GELU)
#   + linear skip: 768 → 512  (bypass the nonlinearities)
#   output = nonlinear_path + skip_path
#
# The residual skip prevents gradient vanishing and lets the
# model learn "how much to compress" rather than forcing it.
# ============================================================

class ResidualProjection(nn.Module):
    """
    3-stage residual projection: in_dim → mid_dim → out_dim
    with a linear skip connection in_dim → out_dim.

    ResNet-style: output = F(x) + W_skip(x)
    """

    def __init__(self, in_dim: int, mid_dim: int, out_dim: int, dropout: float = 0.1):
        super().__init__()

        # Main (nonlinear) path: 3 stages
        self.main = nn.Sequential(
            nn.Linear(in_dim, mid_dim),
            nn.LayerNorm(mid_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mid_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Skip path: linear shortcut (no activation — clean residual)
        self.skip = nn.Linear(in_dim, out_dim, bias=False)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.main(x) + self.skip(x)


class DualResidualProjection(nn.Module):
    """
    Applies independent ResidualProjection to video and text tokens.
    """

    def __init__(self, video_in, text_in, mid_dim, out_dim, dropout=0.1):
        super().__init__()
        self.video_proj = ResidualProjection(video_in, mid_dim, out_dim, dropout)
        self.text_proj  = ResidualProjection(text_in,  mid_dim, out_dim, dropout)

    def forward(self, video_tokens, text_tokens):
        return self.video_proj(video_tokens), self.text_proj(text_tokens)


logger.info("ResidualProjection (3-stage, FIX 5) defined ✓")

## Cell 11 — FIX 4: Modality-Type Embeddings + Sinusoidal PE

In [ ]:
# Cell 11 [v3 FIX — BUG 3]: Modality-Type Embeddings + Sinusoidal PE
# ============================================================
# BUG 3 FIX — Double dropout on token embeddings:
#   v2: ModalityTypeEmbedding.forward() applied self.drop()
#       THEN SinusoidalPositionalEncoding.forward() applied
#       its own self.dropout() immediately after.
#       Two consecutive dropouts on the same vector before
#       any learned transformation: effective drop rate becomes
#       1-(1-0.1)^2 ≈ 19% — nearly double the intended rate.
#       This suppresses gradient signal through the embedding
#       layers and hurts training stability.
#
#   v3 FIX: Dropout REMOVED from ModalityTypeEmbedding.
#       A single dropout in SinusoidalPositionalEncoding (p=0.1)
#       is sufficient and is applied once after BOTH modality
#       embedding and PE are added. This is exactly the pattern
#       used in the original BERT and Transformer implementations.
# ============================================================

class ModalityTypeEmbedding(nn.Module):
    """
    Learnable segment embeddings: one vector per modality (video / text).
    Added element-wise to each token before sinusoidal PE.

    Analogous to BERT's token_type_embeddings (sentence A vs B).

    NOTE: No dropout here — a single dropout is applied by
    SinusoidalPositionalEncoding after both this embedding and
    the PE are summed. Applying dropout in both places compounds
    the drop rate and destabilises early training (BUG 3 fix).
    """

    MODALITY_VIDEO = 0
    MODALITY_TEXT  = 1

    def __init__(self, d_model: int):
        super().__init__()
        self.embed = nn.Embedding(2, d_model)          # 0=video, 1=text
        nn.init.normal_(self.embed.weight, std=0.02)   # same init as BERT
        # NO self.drop — dropout is owned by SinusoidalPositionalEncoding

    def forward(
        self,
        video_tokens: torch.Tensor,   # (B, Lv, D)
        text_tokens:  torch.Tensor,   # (B, Lt, D)
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, Lv, D = video_tokens.shape
        _,  Lt, _ = text_tokens.shape

        vid_type = torch.full((B, Lv), self.MODALITY_VIDEO, dtype=torch.long,
                              device=video_tokens.device)
        txt_type = torch.full((B, Lt), self.MODALITY_TEXT,  dtype=torch.long,
                              device=text_tokens.device)

        # Add modality embedding — NO dropout (BUG 3 fix)
        video_tokens = video_tokens + self.embed(vid_type)   # (B, Lv, D)
        text_tokens  = text_tokens  + self.embed(txt_type)   # (B, Lt, D)
        return video_tokens, text_tokens


class SinusoidalPositionalEncoding(nn.Module):
    """
    Fixed sine-cosine positional encoding (Vaswani 2017).
    Owns the SINGLE dropout applied after modality embed + PE are both added.
    dropout=0.1 is applied once here and nowhere else in the embedding stack.
    """

    def __init__(self, d_model: int, max_len: int = 4096, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))   # (1, max_len, D)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x already has modality embed added upstream
        # Add PE then apply the single dropout (BUG 3 fix)
        return self.dropout(x + self.pe[:, :x.size(1), :])


logger.info('ModalityTypeEmbedding (no dropout, BUG 3 fixed) + SinusoidalPE defined ✓')


## Cell 12 — FIX 1 + FIX 3: Bidirectional Cross-Modal Fusion (no causal mask)

In [ ]:
# Cell 12 [v3 FIX — BUG 1]: BidirectionalCrossModalFusion
# ============================================================
# BUG 1 FIX — Stale key/value norms in cross-attention:
#   v2 forward():
#     v_norm = norm_v1(video)           # computed BEFORE v2t update
#     t_norm = norm_t1(text)
#     v2t, _ = v2t_cross(Q=v_norm, K=t_norm, V=t_norm)
#     video  = video + drop(v2t)        # video UPDATED
#     t2v, _ = t2v_cross(Q=t_norm,      # STALE t_norm reused
#                         K=v_norm,     # STALE v_norm reused (pre-update!)
#                         V=v_norm)
#   text attends to video BEFORE the v2t update — so the
#   information just written into video is invisible to text.
#   This breaks the intended bidirectional update semantics.
#
#   v3 FIX: Recompute norms AFTER each stream update:
#     v_norm2 = norm_v1(video)   # re-norm with UPDATED video
#     t_norm2 = norm_t1(text)    # re-norm with current text
#     t2v, _  = t2v_cross(Q=t_norm2, K=v_norm2, V=v_norm2)
#   Each stream sees the most recent state of the other.
# ============================================================

class BidirectionalCrossModalFusionBlock(nn.Module):
    """
    One bidirectional fusion block with fresh norms after each update.

    Order of operations:
      1. video ← CrossAttn(Q=norm(video), K=norm(text),  V=norm(text))  + residual
      2. text  ← CrossAttn(Q=norm(text),  K=norm(video), V=norm(video)) + residual
         [norms recomputed after step 1 — BUG 1 fix]
      3. video ← SelfAttn(norm(video)) + residual  (bidirectional, no mask)
      4. text  ← SelfAttn(norm(text))  + residual
      5. video ← FFN(norm(video))      + residual
      6. text  ← FFN(norm(text))       + residual
    """

    def __init__(self, d_model: int, num_heads: int, ffn_dim: int, dropout: float = 0.1):
        super().__init__()

        self.v2t_cross  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.t2v_cross  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.video_self = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.text_self  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)

        self.video_ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )
        self.text_ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )

        # Eight dedicated LayerNorms — one per sub-layer per stream (Pre-LN pattern)
        self.norm_v_cross = nn.LayerNorm(d_model)   # video: before v2t cross-attn (query)
        self.norm_v_ctx   = nn.LayerNorm(d_model)   # video: re-normed as K/V for t2v (after v2t update)
        self.norm_v_self  = nn.LayerNorm(d_model)   # video: before video self-attn
        self.norm_v_ffn   = nn.LayerNorm(d_model)   # video: before video FFN
        self.norm_t_cross = nn.LayerNorm(d_model)   # text:  before t2v cross-attn (query)
        self.norm_t_ctx   = nn.LayerNorm(d_model)   # text:  as K/V for v2t cross-attn
        self.norm_t_self  = nn.LayerNorm(d_model)   # text:  before text self-attn
        self.norm_t_ffn   = nn.LayerNorm(d_model)   # text:  before text FFN
        self.drop         = nn.Dropout(dropout)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(
        self,
        video: torch.Tensor,                                      # (B, Lv, D)
        text:  torch.Tensor,                                      # (B, Lt, D)
        text_key_padding_mask:  Optional[torch.Tensor] = None,   # (B, Lt) bool
        video_key_padding_mask: Optional[torch.Tensor] = None,   # (B, Lv) bool
    ) -> Tuple[torch.Tensor, torch.Tensor]:

        # ── Step 1: video attends to text (v2t cross-attention) ─────
        v2t, _ = self.v2t_cross(
            query=self.norm_v_cross(video),    # video queries (pre-norm)
            key=self.norm_t_ctx(text),         # text keys
            value=self.norm_t_ctx(text),       # text values
            key_padding_mask=text_key_padding_mask,
        )
        video = video + self.drop(v2t)         # video is NOW updated

        # ── Step 2: text attends to UPDATED video (t2v cross-attention) ──
        # norm_v_ctx re-applied to post-update video — text sees fresh state.
        # norm_t_cross applied to text as query (each norm is independent).
        t2v, _ = self.t2v_cross(
            query=self.norm_t_cross(text),     # text queries (pre-norm)
            key=self.norm_v_ctx(video),        # UPDATED video keys   ← correct
            value=self.norm_v_ctx(video),      # UPDATED video values ← correct
            key_padding_mask=video_key_padding_mask,
        )
        text = text + self.drop(t2v)           # text is NOW updated

        # ── Step 3: within-modality self-attention (bidirectional, no mask) ──
        v_s = self.norm_v_self(video)
        v_self, _ = self.video_self(query=v_s, key=v_s, value=v_s)
        video = video + self.drop(v_self)

        t_s = self.norm_t_self(text)
        t_self, _ = self.text_self(query=t_s, key=t_s, value=t_s)
        text = text + self.drop(t_self)

        # ── Step 4: FFN — each stream uses its own dedicated FFN norm ──
        video = video + self.video_ffn(self.norm_v_ffn(video))   # ← norm_v_ffn (was wrong norm_v1)
        text  = text  + self.text_ffn(self.norm_t_ffn(text))     # ← norm_t_ffn (was correct norm_t3)

        return video, text


class BidirectionalCrossModalFusion(nn.Module):
    """Stack of NUM_FUSION_LAYERS bidirectional fusion blocks."""

    def __init__(self, d_model, num_heads, ffn_dim, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            BidirectionalCrossModalFusionBlock(d_model, num_heads, ffn_dim, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, video, text, text_pad_mask=None, video_pad_mask=None):
        for layer in self.layers:
            video, text = layer(video, text, text_pad_mask, video_pad_mask)
        return video, text


logger.info('BidirectionalCrossModalFusion (v4 FINAL — 8 dedicated norms, correct FFN norm, fresh cross-attn norms) defined ✓')


## Cell 13 — FIX 2 + FIX 6: LearnedQuerySpanHead (DETR-style, preserves temporal structure)

In [ ]:
# Cell 13: LearnedQuerySpanHead — DETR-style (FIX 2 + FIX 6)
# ============================================================
# FIX 2 — GAP REMOVED:
#   v1: video_desc = GAP(fused_video)  →  (B, D)  [1 vector]
#   GAP averages all Lv token positions into a single descriptor.
#   This discards ALL temporal positional information — the model
#   can no longer tell WHERE in the video an activity occurs.
#   For temporal grounding this is a fundamental design error.
#
# FIX 6 — LSTM REPLACED:
#   v1: LSTM autoregressively predicted spans one-by-one,
#   trained with teacher forcing (feeding ground truth prev span)
#   but run freely at inference (feeding its own pred prev span).
#   This teacher-forcing / free-run mismatch (exposure bias)
#   causes cascading prediction errors.
#
# v2 FIX: DETR-style learned query head:
#   NUM_QUERIES learnable query vectors Q_i ∈ R^D (i=0…K-1)
#   Each query attends to ALL Lv video tokens in parallel:
#     attn_i = softmax(Q_i · K_video^T / √d)
#     span_i = MLP(attn_i · V_video)   → (start_i, end_i)
#   + confidence head:  conf_i = sigmoid(MLP(attn_i · V_video))
#
#   Benefits:
#   1. Temporal structure preserved: each query attends to
#      specific time regions, learning query specialisation.
#   2. No teacher forcing: all K spans predicted in parallel.
#   3. No exposure bias: training = inference.
#   4. Queries compete via Hungarian matching loss (DETR-style).
# ============================================================

class LearnedQuerySpanHead(nn.Module):
    """
    DETR-style span prediction head.

    Inputs:
        fused_video_tokens : (B, Lv, D)  — temporal structure preserved
        text_context       : (B, D)      — BERT [CLS] as text conditioning

    Learned parameters:
        query_embed        : (NUM_QUERIES, D)  — K learnable query slots

    Output:
        start_preds  : (B, NUM_QUERIES) ∈ [0,1]
        end_preds    : (B, NUM_QUERIES) ∈ [0,1]
        conf_logits  : (B, NUM_QUERIES)  raw logit
    """

    def __init__(
        self,
        d_model:     int,
        text_dim:    int,
        num_queries: int,
        num_heads:   int = 8,
        mlp_hidden:  List[int] = [256, 128],
        dropout:     float = 0.1,
    ):
        super().__init__()
        self.num_queries = num_queries

        # K learnable query vectors
        self.query_embed = nn.Embedding(num_queries, d_model)
        nn.init.normal_(self.query_embed.weight, std=0.02)

        # Text context projection → same dim as queries
        self.text_ctx_proj = nn.Linear(text_dim, d_model)

        # Query-to-video cross-attention
        # Q = learned queries (conditioned on text),  K/V = fused video tokens
        # Full bidirectional attention — NO causal mask
        self.query_cross_attn = nn.MultiheadAttention(
            embed_dim   = d_model,
            num_heads   = num_heads,
            dropout     = dropout,
            batch_first = True,
        )
        self.query_norm = nn.LayerNorm(d_model)

        # Query self-attention (allow queries to communicate)
        self.query_self_attn = nn.MultiheadAttention(
            embed_dim   = d_model,
            num_heads   = num_heads,
            dropout     = dropout,
            batch_first = True,
        )
        self.query_self_norm = nn.LayerNorm(d_model)

        # Span regression MLP (sigmoid output → normalised timestamps)
        span_layers = []; prev = d_model
        for h in mlp_hidden:
            span_layers += [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        span_layers += [nn.Linear(prev, 2), nn.Sigmoid()]   # 2 outputs: start, end
        self.span_mlp = nn.Sequential(*span_layers)

        # Confidence head (binary: is this a valid span?)
        self.conf_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(
        self,
        fused_video: torch.Tensor,             # (B, Lv, D)
        text_cls:    torch.Tensor,             # (B, text_dim) — BERT pooler
        video_pad_mask: Optional[torch.Tensor] = None,  # (B, Lv) bool
    ) -> Dict[str, torch.Tensor]:
        B = fused_video.shape[0]

        # Expand learned queries for the batch
        query_idx  = torch.arange(self.num_queries, device=fused_video.device)
        queries    = self.query_embed(query_idx).unsqueeze(0).expand(B, -1, -1)  # (B, K, D)

        # Condition queries on text context
        text_ctx   = self.text_ctx_proj(text_cls).unsqueeze(1)  # (B, 1, D)
        queries    = queries + text_ctx   # broadcast text context to all queries

        # ── Query→Video cross-attention (no causal mask) ──────
        # Each query selectively attends to different temporal regions
        q_norm    = self.query_norm(queries)
        attended, _ = self.query_cross_attn(
            query=q_norm,
            key=fused_video,
            value=fused_video,
            key_padding_mask=video_pad_mask,
        )   # (B, K, D)
        queries = queries + attended

        # ── Query self-attention (no causal mask) ─────────────
        q2 = self.query_self_norm(queries)
        q_self, _ = self.query_self_attn(query=q2, key=q2, value=q2)
        queries = queries + q_self   # (B, K, D)

        # ── Predict spans and confidence from each query ──────
        spans       = self.span_mlp(queries)              # (B, K, 2)  ∈ [0,1]
        conf_logits = self.conf_head(queries).squeeze(-1)  # (B, K)

        # Enforce start ≤ end (soft constraint)
        start = torch.min(spans[:,:,0], spans[:,:,1])   # (B, K)
        end   = torch.max(spans[:,:,0], spans[:,:,1])   # (B, K)

        return {
            "start_preds": start,        # (B, NUM_QUERIES)
            "end_preds":   end,          # (B, NUM_QUERIES)
            "conf_logits": conf_logits,  # (B, NUM_QUERIES) raw logit
        }


logger.info("LearnedQuerySpanHead (DETR-style, FIX 2 + FIX 6) defined ✓")
logger.info("  → GAP removed; all Lv tokens preserved; no LSTM teacher-forcing mismatch")

## Cell 14 — Full Temporal Grounding Model v2

In [ ]:
# Cell 14: TemporalGroundingModelV2 — ViT backbone only
# ============================================================
# All bug fixes (BUG 1–5) retained. Swin backbone removed.
# Video encoder is always VideoViT.
# VIDEO_FEAT_DIM=768 = VIT_EMBED_DIM — no shape changes downstream.
# ============================================================

class TemporalGroundingModelV2(nn.Module):
    """
    Video Temporal Grounding Model — ViT backbone edition.

    All fixes vs v2:
      BUG 1: Stale norms in cross-attn  → fresh norm_v2/t1 after each update
      BUG 2: Float loss accumulation    → tensor-init total_l1/giou
      BUG 3a: Double dropout            → dropout removed from ModalityTypeEmbedding
      BUG 3b: BERT overfitting          → TextEncoder(freeze_layers=6)
      BUG 5: cls_emb distribution gap   → fused_text[:,0,:] used instead
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        # 1. Video Encoder — VideoViT
        self.video_encoder = build_video_vit(cfg)

        # 2. Text Encoder — BERT with selective layer freezing
        self.text_encoder = TextEncoder(cfg.BERT_MODEL, freeze_layers=cfg.BERT_FREEZE_LAYERS)

        # 3. 3-stage residual projection (768→512)
        self.projection = DualResidualProjection(
            video_in=cfg.VIDEO_FEAT_DIM,
            text_in =cfg.TEXT_FEAT_DIM,
            mid_dim =cfg.PROJ_MID_DIM,
            out_dim =cfg.SHARED_DIM,
            dropout =0.1,
        )

        # 4. Modality-type embeddings
        self.modality_embed = ModalityTypeEmbedding(cfg.SHARED_DIM)

        # 5. Sinusoidal PE
        self.pos_encoding = SinusoidalPositionalEncoding(
            d_model=cfg.SHARED_DIM, max_len=4096, dropout=cfg.FUSION_DROP,
        )

        # 6. Bidirectional fusion
        self.fusion = BidirectionalCrossModalFusion(
            d_model   =cfg.SHARED_DIM,
            num_heads =cfg.FUSION_HEADS,
            ffn_dim   =cfg.FUSION_FFN_DIM,
            num_layers=cfg.NUM_FUSION_LAYERS,
            dropout   =cfg.FUSION_DROP,
        )

        # 7. DETR-style span head
        self.span_head = LearnedQuerySpanHead(
            d_model    =cfg.SHARED_DIM,
            text_dim   =cfg.SHARED_DIM,
            num_queries=cfg.NUM_QUERIES,
            num_heads  =cfg.FUSION_HEADS,
            mlp_hidden =cfg.MLP_HIDDEN_DIMS,
            dropout    =cfg.MLP_DROP,
        )

        self._init_new_heads()

    def _init_new_heads(self):
        for m in self.projection.modules():
            if isinstance(m, nn.Linear):
                trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def encode(
        self, video, input_ids, attention_mask, token_type_ids
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            fused_video : (B, Lv, SHARED_DIM)
            fused_text  : (B, Lt, SHARED_DIM)
            fused_cls   : (B, SHARED_DIM)  — [CLS] token post-fusion
        """
        raw_video         = self.video_encoder(video)
        raw_text, _       = self.text_encoder(input_ids, attention_mask, token_type_ids)
        video_proj, text_proj = self.projection(raw_video, raw_text)
        video_proj, text_proj = self.modality_embed(video_proj, text_proj)
        video_proj = self.pos_encoding(video_proj)
        text_proj  = self.pos_encoding(text_proj)
        text_pad_mask = (attention_mask == 0)
        fused_video, fused_text = self.fusion(
            video=video_proj, text=text_proj,
            text_pad_mask=text_pad_mask, video_pad_mask=None,
        )
        fused_cls = fused_text[:, 0, :]
        return fused_video, fused_text, fused_cls

    def forward(
        self,
        video:          torch.Tensor,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        duration:       Optional[torch.Tensor] = None,
        training:       bool = True,
    ) -> Dict[str, Any]:

        fused_video, fused_text, fused_cls = self.encode(
            video, input_ids, attention_mask, token_type_ids
        )

        out = self.span_head(
            fused_video=fused_video,
            text_cls   =fused_cls,
        )

        if training:
            return out

        B = video.shape[0]
        spans_batch = []
        for b in range(B):
            dur = duration[b].item() if duration is not None else 1.0
            conf_probs = torch.sigmoid(out['conf_logits'][b])
            spans = []
            for k in range(self.cfg.NUM_QUERIES):
                if conf_probs[k].item() >= self.cfg.CONF_THRESHOLD:
                    ts = out['start_preds'][b, k].item() * dur
                    te = out['end_preds'][b,   k].item() * dur
                    if te > ts:
                        spans.append((round(ts, 2), round(te, 2)))
            if not spans:
                best_k = conf_probs.argmax().item()
                ts = out['start_preds'][b, best_k].item() * dur
                te = out['end_preds'][b,   best_k].item() * dur
                te = max(te, ts + 0.1)
                spans = [(round(ts, 2), round(te, 2))]
            spans.sort(key=lambda s: s[0])
            spans_batch.append(spans)

        return {'spans': spans_batch}


logger.info('TemporalGroundingModelV2 (ViT backbone, all bugs fixed) defined ✓')


## Cell 15 — Dataset & DataLoader (CharadesSTA — unchanged)

In [ ]:
# Cell 15: CharadesSTA Dataset & DataLoader (v4 — clip-consistent augmentation)
# ============================================================
import torchvision.transforms.functional as TF

class CharadesSTA(Dataset):
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, json_path, video_dir, tokenizer, cfg, split="train"):
        self.video_dir = video_dir; self.tokenizer = tokenizer
        self.cfg = cfg; self.split = split
        with open(json_path, "r") as f: raw = json.load(f)
        self.samples = []
        for entry in raw:
            vid_id = entry["video_id"]; dur = float(entry["duration"])
            for anno in entry["annotations"]:
                self.samples.append({
                    "video_id": vid_id, "duration": dur,
                    "sentence": anno["sentence"], "timestamps": [anno["timestamp"]],
                })
        # Augmentation is now applied CLIP-consistently inside _load_video_frames.
        # self.split is stored so _load_video_frames can branch on train vs val/test.
        logger.info(f"[{split}] {len(self.samples)} samples loaded.")

    def _load_video_frames(self, video_id):
        cv2.setNumThreads(0)
        cap = cv2.VideoCapture(str(self.video_dir / video_id))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        total = max(total, self.cfg.NUM_FRAMES)
        # Use a list (ordered) not a set — seeking is by position  ← 8 spaces
        indices = np.linspace(0, total - 1, self.cfg.NUM_FRAMES, dtype=int).tolist()

        frames = []
        last_frame = np.zeros((self.cfg.FRAME_H, self.cfg.FRAME_W, 3), dtype=np.uint8)

        # Direct seek instead of sequential scan              ← 8 spaces
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(
                    cv2.cvtColor(frame, cv2.COLOR_BGR2RGB),
                    (self.cfg.FRAME_W, self.cfg.FRAME_H),
                    interpolation=cv2.INTER_LINEAR,
                )
                frames.append(frame)
                last_frame = frame
            else:
                frames.append(last_frame.copy())
        cap.release()

        # Pad if still short                                  ← 8 spaces
        while len(frames) < self.cfg.NUM_FRAMES:
            frames.append(frames[-1].copy() if frames else last_frame)

        # (T, H, W, C) → (C, T, H, W)                       ← 8 spaces
        video_np = np.stack(frames, axis=0).astype(np.float32) / 255.0
        video_t  = torch.from_numpy(video_np).permute(3, 0, 1, 2)
        ...  # rest unchanged
        if self.split == 'train':
            n_ch, n_frames, n_h, n_w = video_t.shape
            crop_params = T.RandomResizedCrop.get_params(
                video_t[:, 0], scale=(0.7, 1.0), ratio=(3/4, 4/3),
            )
            i_c, j_c, h_c, w_c = crop_params
            frames_aug = []
            for t in range(n_frames):
                f = video_t[:, t]
                f = TF.resized_crop(f, i_c, j_c, h_c, w_c,
                                (self.cfg.FRAME_H, self.cfg.FRAME_W),
                                interpolation=T.InterpolationMode.BILINEAR)
                frames_aug.append(f)
            video_t = torch.stack(frames_aug, dim=1)

            if random.random() < 0.5:
                video_t = torch.flip(video_t, dims=[3])

            cj = T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)
            cpu_rng_state  = torch.get_rng_state()
            cuda_rng_state = torch.cuda.get_rng_state() if torch.cuda.is_available() else None
            frames_cj = []
            for t in range(video_t.shape[1]):
                torch.set_rng_state(cpu_rng_state)
                if cuda_rng_state is not None:
                    torch.cuda.set_rng_state(cuda_rng_state)
                frames_cj.append(cj(video_t[:, t]))
            video_t = torch.stack(frames_cj, dim=1)

        else:
            frames_val = []
            for t in range(video_t.shape[1]):
                frames_val.append(TF.resize(
                    video_t[:, t],
                    (self.cfg.FRAME_H, self.cfg.FRAME_W),
                    interpolation=T.InterpolationMode.BILINEAR,
                ))
            video_t = torch.stack(frames_val, dim=1)

        mean = torch.tensor(self.MEAN, dtype=video_t.dtype).view(3, 1, 1, 1)
        std  = torch.tensor(self.STD,  dtype=video_t.dtype).view(3, 1, 1, 1)
        video_t = (video_t - mean) / std
        return video_t

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]; dur = s["duration"]
        video = self._load_video_frames(s["video_id"])
        enc = self.tokenizer(s["sentence"], max_length=self.cfg.MAX_TEXT_LEN, padding="max_length",
                             truncation=True, return_tensors="pt")
        n_valid = min(len(s["timestamps"]), self.cfg.NUM_QUERIES)
        start_norm = torch.zeros(self.cfg.NUM_QUERIES)
        end_norm   = torch.zeros(self.cfg.NUM_QUERIES)
        span_mask  = torch.zeros(self.cfg.NUM_QUERIES)
        for i in range(n_valid):
            ts = s["timestamps"][i]
            start_norm[i] = float(ts[0]) / max(dur, 1e-6)
            end_norm[i]   = float(ts[1]) / max(dur, 1e-6)
            span_mask[i]  = 1.0
        return {
            "video":          video,
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": enc["token_type_ids"].squeeze(0),
            "start_norm":     start_norm,
            "end_norm":       end_norm,
            "span_mask":      span_mask,
            "duration":       torch.tensor(dur, dtype=torch.float32),
        }


def _worker_init_fn(worker_id):
    """
    Called once per DataLoader worker process at startup.
    OpenCV forks badly on Windows — resetting its thread count
    inside the worker prevents the VideoCapture deadlock.
    """
    import cv2
    cv2.setNumThreads(0)          # disable OpenCV's own thread pool inside workers
    import numpy as np
    np.random.seed(SEED + worker_id)


def build_dataloaders(cfg, tokenizer):
    num_workers = cfg.NUM_WORKERS

    kw = dict(num_workers=num_workers, pin_memory=cfg.PIN_MEMORY)

    if num_workers > 0:
        kw["persistent_workers"] = cfg.PERSISTENT_WORKERS
        kw["prefetch_factor"]    = cfg.PREFETCH_FACTOR
        kw["worker_init_fn"]     = _worker_init_fn
        # 'fork' is unavailable on Windows; 'spawn' is safe but slow to start.
        # 'forkserver' is Linux-only. Use spawn explicitly so PyTorch doesn't
        # silently fall back to a broken default on Windows.
        kw["multiprocessing_context"] = "spawn"

    train_ds = CharadesSTA(cfg.TRAIN_JSON, cfg.VIDEO_DIR, tokenizer, cfg, "train")
    val_ds   = CharadesSTA(cfg.VAL_JSON,   cfg.VIDEO_DIR, tokenizer, cfg, "val")
    test_ds  = CharadesSTA(cfg.TEST_JSON,  cfg.VIDEO_DIR, tokenizer, cfg, "test")

    return (
        DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,   shuffle=True,
                   drop_last=True, **kw),
        DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE*2, shuffle=False, **kw),
        DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE*2, shuffle=False, **kw),
    )

## Cell 16 — Loss Function (Hungarian matching for DETR-style head)

In [ ]:
# Cell 16 [BF16 FIX]: temporal_eiou_1d — safe epsilon for BF16
# ============================================================
# BUG: eps=1e-7 squared to 1e-14 which underflows to 0 in BF16
# (BF16 has ~3 decimal digits of mantissa precision at 1e-7).
# C² = 0 → division by zero → NaN gradient → skipped optimizer
# step → wasted batches. Seen as NaN at step 0 in the log.
# FIX: raise eps to 1e-4 and clamp C² to prevent any division
# by a near-zero value regardless of floating-point mode.
# ============================================================

def temporal_eiou_1d(ps, pe, gs, ge):
    """
    Efficient IoU (EIoU) for 1-D temporal segments — BF16-safe.
    eps raised from 1e-7 → 1e-4; C² explicitly clamped.
    """
    # ↓ was 1e-7 — too small for BF16 (squares to ~0, causes NaN)
    eps = 1e-4

    ps = torch.clamp(ps, 0.0, 1.0)
    pe = torch.clamp(pe, 0.0, 1.0)
    pe = torch.max(ps, pe)

    inter = torch.clamp(torch.min(pe, ge) - torch.max(ps, gs), min=0.0)
    union = (pe - ps) + (ge - gs) - inter + eps
    iou   = inter / union

    C  = torch.max(pe, ge) - torch.min(ps, gs) + eps
    # ↓ new: explicit clamp prevents division by near-zero after BF16 rounding
    C2 = torch.clamp(C * C, min=1e-8)

    bc   = (ps + pe) * 0.5
    bc_t = (gs + ge) * 0.5
    centre_penalty = (bc - bc_t) ** 2 / C2

    w   = pe - ps
    w_t = ge - gs
    width_penalty = (w - w_t) ** 2 / C2

    return iou - centre_penalty - width_penalty


# Keep alias so nothing else breaks
temporal_giou_1d = temporal_eiou_1d


@torch.no_grad()
def hungarian_match(
    pred_start, pred_end, gt_start, gt_end,
    lambda_l1=5.0, lambda_iou=2.0,
):
    """
    Hungarian matching using EIoU cost instead of GIoU.

    Cost matrix entry (k, g):
        lambda_l1  * (|ps_k − gs_g| + |pe_k − ge_g|)    L1 endpoint cost
      + lambda_iou * (1 − EIoU(ps_k, pe_k, gs_g, ge_g))  EIoU cost

    FIX: cost matrix is sanitised before linear_sum_assignment to prevent
    ValueError when AMP/optimizer instability after resume produces NaN/inf
    in predictions → cost entries blow up.
    """
    K, G = len(pred_start), len(gt_start)
    cost = torch.zeros(K, G)
    for k in range(K):
        ps = pred_start[k].expand(G)
        pe = pred_end[k].expand(G)
        cost[k] = (
            lambda_l1  * ((ps - gt_start).abs() + (pe - gt_end).abs())
          + lambda_iou * (1.0 - temporal_eiou_1d(ps, pe, gt_start, gt_end))
        )

    cost_np = cost.cpu().numpy()

    # ── FIX: guard NaN/inf before passing to scipy ───────────────────────────
    # Root cause: after checkpoint resume, AMP scaler or Adam momentum buffers
    # are momentarily unstable → model outputs inf/nan → cost matrix blows up
    # → linear_sum_assignment raises "matrix contains invalid numeric entries".
    # nan_to_num clamps to a large finite value so matching still runs and the
    # non-finite grad / loss guards in train_one_epoch discard the bad batch.
    if not np.isfinite(cost_np).all():
        n_nan = int(np.isnan(cost_np).sum())
        n_inf = int(np.isinf(cost_np).sum())
        logger.warning(
            f"[hungarian_match] Non-finite cost matrix entries detected "
            f"(nan={n_nan}, inf={n_inf}, shape={cost_np.shape}). "
            f"Clamping to 1e6 — batch will be skipped by train_one_epoch guard."
        )
        cost_np = np.nan_to_num(cost_np, nan=1e6, posinf=1e6, neginf=-1e6)

    row_ind, col_ind = linear_sum_assignment(cost_np)
    return row_ind.tolist(), col_ind.tolist()


class TemporalGroundingLossV2(nn.Module):
    """
    Temporal grounding loss with Hungarian matching — EIoU edition.

    Replaces GIoU with EIoU (Mohammed et al., 2024) for the IoU term.
    Everything else (BUG 2 fix, tensor-init accumulators, conf BCE) is
    unchanged from v3.

    The returned dict uses the key 'eiou_loss' instead of 'giou_loss'.
    """

    def __init__(self, lambda_l1=5.0, lambda_iou=2.0, lambda_conf=1.0):
        super().__init__()
        self.lambda_l1   = lambda_l1
        self.lambda_iou  = lambda_iou
        self.lambda_conf = lambda_conf
        self.bce = nn.BCEWithLogitsLoss(reduction='mean')

    def forward(
        self,
        start_preds: torch.Tensor,   # (B, K)
        end_preds:   torch.Tensor,   # (B, K)
        conf_logits: torch.Tensor,   # (B, K)
        start_gt:    torch.Tensor,   # (B, K)
        end_gt:      torch.Tensor,   # (B, K)
        span_mask:   torch.Tensor,   # (B, K) 1 = valid GT span
    ) -> Dict[str, torch.Tensor]:

        B, K = start_preds.shape

        total_l1    = start_preds.new_zeros(1)
        total_eiou  = start_preds.new_zeros(1)
        conf_target = torch.zeros_like(conf_logits)
        n_matched   = 0

        for b in range(B):
            valid_idx = span_mask[b].nonzero(as_tuple=True)[0]
            if len(valid_idx) == 0:
                continue

            gs = start_gt[b, valid_idx]
            ge = end_gt[b,   valid_idx]

            pred_i, gt_i = hungarian_match(
                start_preds[b].detach(), end_preds[b].detach(), gs, ge,
                self.lambda_l1, self.lambda_iou,
            )

            for pi, gi in zip(pred_i, gt_i):
                ps, pe     = start_preds[b, pi], end_preds[b, pi]
                gs_i, ge_i = gs[gi], ge[gi]

                total_l1   = total_l1   + (ps - gs_i).abs() + (pe - ge_i).abs()
                total_eiou = total_eiou + (1.0 - temporal_eiou_1d(
                    ps.unsqueeze(0), pe.unsqueeze(0),
                    gs_i.unsqueeze(0), ge_i.unsqueeze(0),
                ))
                conf_target[b, pi] = 1.0
                n_matched += 1

        denom     = max(n_matched, 1)
        l1_loss   = total_l1[0]   / denom
        eiou_loss = total_eiou[0] / denom
        conf_loss = self.bce(conf_logits, conf_target)

        total = (self.lambda_l1  * l1_loss
               + self.lambda_iou * eiou_loss
               + self.lambda_conf * conf_loss)

        return {
            'loss':      total,
            'l1_loss':   l1_loss,
            'eiou_loss': eiou_loss,
            'conf_loss': conf_loss,
        }


logger.info('TemporalGroundingLossV2 (EIoU — centre + width penalty) defined ✓')

## Cell 17 — Evaluation Metric: R@1 IoU (unchanged)

In [ ]:
# Cell 17: Evaluation — R@1 at IoU thresholds (unchanged from v1)
# ============================================================

def temporal_iou(ps, pe, gs, ge):
    inter = max(0.0, min(pe, ge) - max(ps, gs))
    union = (pe - ps) + (ge - gs) - inter
    return inter / max(union, 1e-7)


@torch.no_grad()
def evaluate(model, loader, cfg, device):
    model.eval()
    hits = defaultdict(int); total = 0
    for batch in loader:
        video          = batch["video"].to(device)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        start_gt       = batch["start_norm"]
        end_gt         = batch["end_norm"]
        span_mask      = batch["span_mask"]

        with autocast("cuda", enabled=cfg.AMP):
            # training=True returns raw (start_preds, end_preds, conf_logits) tensors
            # which is exactly what we need for metric computation.
            # model.eval() above disables all nn.Dropout layers independently of
            # the `training` kwarg — so dropout IS correctly off here.
            # Using training=True (raw tensor path) avoids the duration-scaling
            # decode branch and keeps evaluate() self-contained.
            out = model(video, input_ids, attention_mask, token_type_ids, training=True)

        start_preds = out["start_preds"].cpu()
        end_preds   = out["end_preds"].cpu()
        conf_probs  = torch.sigmoid(out["conf_logits"]).cpu()

        for b in range(video.shape[0]):
            # Best = highest confidence prediction
            best_idx = conf_probs[b].argmax().item()
            ps, pe   = start_preds[b, best_idx].item(), end_preds[b, best_idx].item()
            valid_idx = span_mask[b].nonzero(as_tuple=True)[0]
            if not len(valid_idx): continue
            fi       = valid_idx[0].item()
            gs, ge   = start_gt[b, fi].item(), end_gt[b, fi].item()
            iou      = temporal_iou(ps, pe, gs, ge)
            for t in cfg.EVAL_METRICS_IOU:
                if iou >= t: hits[t] += 1
            total += 1

    return {f"R@1_IoU{t}": hits[t] / max(total,1) * 100.0 for t in cfg.EVAL_METRICS_IOU}

logger.info("Evaluation (R@1 IoU) defined ✓")

## Cell 18 — Optimizer & Scheduler

In [ ]:
# Cell 18: Optimizer, LR Scheduler & Scaler — ViT-only
# ============================================================
# 2 parameter groups:
#   BERT encoder   → LR_BERT
#   All other heads → LR_HEAD (ViT, projection, fusion, span head)
# ============================================================

from torch.utils.tensorboard import SummaryWriter


def build_optimizer_and_scheduler(model, cfg, total_steps):
    no_decay = {"bias", "norm", "LayerNorm", "layernorm",
                "pe", "modality_embed", "pos_embed", "cls_token"}

    def split(named_params):
        d, nd = [], []
        for n, p in named_params:
            (nd if any(x in n for x in no_decay) else d).append(p)
        return d, nd

    bert_d,  bert_nd  = split(model.text_encoder.named_parameters())

    bert_ids = {id(p) for p in model.text_encoder.parameters()}
    head_d, head_nd = [], []
    for n, p in model.named_parameters():
        if id(p) in bert_ids: continue
        (head_nd if any(x in n for x in no_decay) else head_d).append(p)

    optimizer = optim.AdamW([
        {"params": bert_d,  "lr": cfg.LR_BERT, "weight_decay": cfg.WEIGHT_DECAY},
        {"params": bert_nd, "lr": cfg.LR_BERT, "weight_decay": 0.0},
        {"params": head_d,  "lr": cfg.LR_HEAD, "weight_decay": cfg.WEIGHT_DECAY},
        {"params": head_nd, "lr": cfg.LR_HEAD, "weight_decay": 0.0},
    ], betas=(0.9, 0.999), eps=1e-8)

    warmup  = max(cfg.WARMUP_EPOCHS * max(total_steps // cfg.NUM_EPOCHS, 1), 1)
    cosine_steps = max(total_steps - warmup, 1)
    scheduler = SequentialLR(optimizer, [
        LinearLR(optimizer, start_factor=1e-4, end_factor=1.0, total_iters=warmup),
        CosineAnnealingLR(optimizer, T_max=cosine_steps, eta_min=1e-7),
    ], milestones=[warmup])

    try:
        scaler = GradScaler("cuda", enabled=cfg.AMP)
    except TypeError:
        scaler = GradScaler(enabled=cfg.AMP)
    total_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"[ViT] Trainable params: {total_p/1e6:.2f} M")
    return optimizer, scheduler, scaler


logger.info("Optimizer / Scheduler builder (ViT-only, 2 param groups) defined ✓")


# Cell 18b: Early Stopping Utility


In [ ]:
# Cell 18b: Early Stopping Utility
# ============================================================
# Monitors a validation metric (R@1 IoU@0.5) and halts training
# when no improvement is seen for `patience` consecutive eval cycles.
#
# Why early stopping here?
#   CharadesSTA has ~10K training samples — the model can overfit
#   long before NUM_EPOCHS=20 completes, especially for ViT which
#   lacks Swin's inductive locality bias.  Early stopping recovers
#   training time and selects the best generalising checkpoint
#   without hyperparameter search for NUM_EPOCHS.
#
# Parameters:
#   patience    : epochs with no improvement before stopping.
#                 Recommended: 5 (covers ≥2 eval cycles at every-2-epoch eval).
#   min_delta   : minimum absolute improvement counted as "better".
#                 0.1% is meaningful for R@1 IoU@0.5.
#   mode        : 'max'  (higher metric is better — e.g. R@1)
#                 'min'  (lower is better — e.g. loss)
#   restore_best: if True, model weights are rolled back to the best
#                 seen state when stopping is triggered.
# ============================================================

import copy

class EarlyStopping:
    """
    Tracks a monitored metric and signals when training should stop.

    Usage:
        es = EarlyStopping(patience=5, min_delta=0.1, mode='max')
        for epoch in ...:
            val_metric = evaluate(...)
            if es(val_metric, model):
                break        # es.best_weights already restored into model
    """

    def __init__(
        self,
        patience:     int   = 5,
        min_delta:    float = 0.1,
        mode:         str   = 'max',
        restore_best: bool  = True,
        verbose:      bool  = True,
    ):
        assert mode in ('max', 'min'), "mode must be 'max' or 'min'"
        self.patience      = patience
        self.min_delta     = min_delta
        self.mode          = mode
        self.restore_best  = restore_best
        self.verbose       = verbose

        self.best_score    = None        # best value seen so far
        self.best_weights  = None        # deep-copy of model.state_dict() at best
        self.counter       = 0           # consecutive non-improving epochs
        self.stopped_epoch = None        # epoch index at which stopping fired
        self.triggered     = False       # True once stop condition met

    # ── Internal comparison ────────────────────────────────────
    def _is_improvement(self, score: float) -> bool:
        if self.best_score is None:
            return True
        if self.mode == 'max':
            return score > self.best_score + self.min_delta
        else:
            return score < self.best_score - self.min_delta

    # ── Call once per evaluation epoch ────────────────────────
    def __call__(self, score: float, model: nn.Module, epoch: int = -1) -> bool:
        """
        Returns True  → stop training now.
        Returns False → continue training.
        """
        if self._is_improvement(score):
            self.best_score   = score
            self.counter      = 0
            if self.restore_best:
                # Deep-copy keeps weights on the same device
                self.best_weights = copy.deepcopy(model.state_dict())
            if self.verbose:
                logger.info(
                    f"[EarlyStopping] ✓ New best score = {score:.4f}"
                    f"  (epoch {epoch+1})"
                )
        else:
            self.counter += 1
            if self.verbose:
                logger.info(
                    f"[EarlyStopping] No improvement. "
                    f"Counter {self.counter}/{self.patience}  "
                    f"(best={self.best_score:.4f}, current={score:.4f})"
                )

        if self.counter >= self.patience:
            self.triggered     = True
            self.stopped_epoch = epoch
            if self.restore_best and self.best_weights is not None:
                model.load_state_dict(self.best_weights)
                if self.verbose:
                    logger.info(
                        f"[EarlyStopping] ✗ Triggered at epoch {epoch+1}. "
                        f"Best weights restored (best score={self.best_score:.4f})."
                    )
            return True   # signal: stop

        return False   # signal: continue


# ── Config additions for early stopping ──────────────────────────────
# Add these to Config (Cell 3) or override here for convenience:
cfg.ES_PATIENCE   = 5      # eval cycles without improvement → stop
cfg.ES_MIN_DELTA  = 0.1    # minimum R@1 IoU@0.5 (%) improvement to count
cfg.ES_MODE       = 'max'  # R@1 is "higher is better"

logger.info(
    f"[EarlyStopping] Ready.  patience={cfg.ES_PATIENCE}, "
    f"min_delta={cfg.ES_MIN_DELTA}%, mode={cfg.ES_MODE}"
)

## Cell 19 — Training Loop

In [ ]:
# Cell 19: Training Loop
# ============================================================

from tqdm.auto import tqdm


def train_one_epoch(model, loader, optimizer, scheduler, scaler,
                    criterion, cfg, device, epoch, writer, global_step):
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    n_batches  = 0

    pbar = tqdm(loader, desc=f"[ViT] Epoch {epoch+1}/{cfg.NUM_EPOCHS}", leave=False)
    for step, batch in enumerate(pbar):
        video          = batch["video"].to(device, non_blocking=True)
        input_ids      = batch["input_ids"].to(device, dtype=torch.long, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, dtype=torch.long, non_blocking=True)
        token_type_ids = batch["token_type_ids"].to(device, dtype=torch.long, non_blocking=True)
        start_gt       = batch["start_norm"].to(device, non_blocking=True)
        end_gt         = batch["end_norm"].to(device, non_blocking=True)
        span_mask      = batch["span_mask"].to(device, non_blocking=True)

        amp_device = "cuda" if device.type == "cuda" else "cpu"
        with autocast(amp_device, enabled=cfg.AMP and device.type == "cuda"):
            out    = model(video, input_ids, attention_mask, token_type_ids, training=True)
            losses = criterion(
                out["start_preds"], out["end_preds"], out["conf_logits"],
                start_gt, end_gt, span_mask,
            )
            loss = losses["loss"] / cfg.GRAD_ACCUM

        if not torch.isfinite(losses["loss"]):
            logger.warning(
                f"[train] Non-finite loss={losses['loss'].item()} at step {global_step}. "
                f"Skipping batch."
            )
            optimizer.zero_grad()
            continue

        scaler.scale(loss).backward()

        is_accum_step = (step + 1) % cfg.GRAD_ACCUM == 0
        is_last_step  = (step + 1) == len(loader)

        if is_accum_step or is_last_step:
            if device.type == "cuda" and global_step % 50 == 0:
                writer.add_scalar("system/vram_allocated_gb",
                                  torch.cuda.memory_allocated(0) / 1e9, global_step)
                writer.add_scalar("system/vram_reserved_gb",
                                  torch.cuda.memory_reserved(0) / 1e9, global_step)

            scaler.unscale_(optimizer)
            total_norm = nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_GRAD)

            if not torch.isfinite(total_norm):
                logger.warning(
                    f"[train] Non-finite grad norm={total_norm:.2f} at step {global_step}. "
                    f"Skipping optimizer step."
                )
                optimizer.zero_grad()
                scaler.update()
            else:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

                writer.add_scalar("vit/train/loss",      losses["loss"].item(),      global_step)
                writer.add_scalar("vit/train/l1_loss",   losses["l1_loss"].item(),   global_step)
                writer.add_scalar("Loss/eiou",           losses["eiou_loss"].item(), global_step)
                writer.add_scalar("vit/train/conf_loss", losses["conf_loss"].item(), global_step)
                writer.add_scalar("vit/lr/bert",         optimizer.param_groups[0]["lr"], global_step)
                writer.add_scalar("vit/lr/head",         optimizer.param_groups[2]["lr"], global_step)
                global_step += 1

        total_loss += losses["loss"].item()
        n_batches  += 1
        pbar.set_postfix({
            "loss": f"{losses['loss'].item():.4f}",
            "eiou": f"{losses['eiou_loss'].item():.4f}",
        })

    return total_loss / max(n_batches, 1), global_step


def save_checkpoint(model, optimizer, scheduler, scaler, epoch, metrics, path):
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state":    scaler.state_dict(),
        "metrics":         metrics,
    }, path)
    logger.info(f"[ViT] Checkpoint → {path}")


def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None,
                    device=torch.device("cpu")):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    if optimizer: optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler: scheduler.load_state_dict(ckpt["scheduler_state"])
    if scaler:    scaler.load_state_dict(ckpt["scaler_state"])
    logger.info(f"[ViT] Resumed from epoch {ckpt['epoch']+1}")
    return ckpt["epoch"]


logger.info("Training loop utilities defined ✓")


## Cell 20 — Model Instantiation, Parameter Count & Sanity Check

In [ ]:
# Cell 20: Model Instantiation & Forward Sanity Check — ViT
# ============================================================

model = TemporalGroundingModelV2(cfg).to(DEVICE)
tokenizer = model.text_encoder.tokenizer
criterion = TemporalGroundingLossV2(lambda_l1=5.0, lambda_iou=2.0, lambda_conf=1.0).to(DEVICE)

def count_params(m):
    return (sum(p.numel() for p in m.parameters()),
            sum(p.numel() for p in m.parameters() if p.requires_grad))

t, tr       = count_params(model)
t_vid, _    = count_params(model.video_encoder)
t_bert, _   = count_params(model.text_encoder.bert)
t_fusion, _ = count_params(model.fusion)
t_head, _   = count_params(model.span_head)
logger.info("=" * 60)
logger.info(f"[ViT] Total parameters        : {t/1e6:.2f} M")
logger.info(f"[ViT] Trainable parameters    : {tr/1e6:.2f} M")
logger.info(f"  Video encoder (ViT)         : {t_vid/1e6:.2f} M")
logger.info(f"  BERT encoder                : {t_bert/1e6:.2f} M")
logger.info(f"  Bidirectional fusion        : {t_fusion/1e6:.2f} M")
logger.info(f"  LearnedQuerySpanHead        : {t_head/1e6:.2f} M")
logger.info("=" * 60)

# Forward sanity check
_vid  = torch.randn(1, 3, cfg.NUM_FRAMES, cfg.FRAME_H, cfg.FRAME_W).to(DEVICE)
_text = tokenizer("a person is walking", max_length=cfg.MAX_TEXT_LEN,
                   padding="max_length", return_tensors="pt")
model.eval()
with torch.no_grad():
    with autocast("cuda", enabled=cfg.AMP):
        _out = model(_vid,
                     _text["input_ids"].to(DEVICE),
                     _text["attention_mask"].to(DEVICE),
                     _text["token_type_ids"].to(DEVICE),
                     training=True)
    fv, ft, fcls = model.encode(
        _vid,
        _text["input_ids"].to(DEVICE),
        _text["attention_mask"].to(DEVICE),
        _text["token_type_ids"].to(DEVICE),
    )
    logger.info(f"[ViT] start_preds : {_out['start_preds'].shape}")
    logger.info(f"[ViT] fused_video : {fv.shape}  ({fv.shape[1]} temporal tokens)")
    logger.info(f"[ViT] fused_cls   : {fcls.shape}")
model.train()
logger.info("[ViT] Instantiation & sanity check [OK]")


# Cell 20b: Comprehensive Model Profiler -- Swin vs ViT


In [ ]:
# Cell 20b: Model Profiler — ViT
# ============================================================
# Measures four axes:
#   1. Space complexity     -- parameter count (M)
#   2. Computational cost   -- GFLOPs via thop (video encoder only)
#   3. Time complexity      -- inference latency averaged over 20 runs (ms)
#   4. VRAM footprint       -- peak memory per forward pass (MB)
#
# Install thop once:  pip install thop
# ============================================================
import time as _time_prof

try:
    from thop import profile as thop_profile
    _THOP = True
except ImportError:
    _THOP = False
    logger.warning("[profiler] thop not found. FLOPs skipped. Run: pip install thop")

PROFILE_RESULTS = {}

_vid_p = torch.randn(1, 3, cfg.NUM_FRAMES, cfg.FRAME_H, cfg.FRAME_W).to(DEVICE)
_enc_p = tokenizer(
    "a person picks up an object from the table",
    max_length=cfg.MAX_TEXT_LEN, padding="max_length", return_tensors="pt"
)
_ids_p   = _enc_p["input_ids"].to(DEVICE)
_amask_p = _enc_p["attention_mask"].to(DEVICE)
_ttids_p = _enc_p["token_type_ids"].to(DEVICE)


def profile_model(bname: str, mdl: nn.Module) -> dict:
    mdl.eval()
    res = {"name": bname}

    total, trainable = count_params(mdl)
    vid_p, _         = count_params(mdl.video_encoder)
    res["params_total_M"]     = round(total     / 1e6, 2)
    res["params_trainable_M"] = round(trainable / 1e6, 2)
    res["params_video_enc_M"] = round(vid_p     / 1e6, 2)

    if _THOP:
        try:
            flops, _ = thop_profile(mdl.video_encoder, inputs=(_vid_p,), verbose=False)
            res["gflops_video_enc"] = round(flops / 1e9, 2)
        except Exception as e:
            res["gflops_video_enc"] = None
            logger.warning(f"[profiler/{bname}] thop error: {e}")
    else:
        res["gflops_video_enc"] = None

    N_WARM, N_RUN = 5, 20
    with torch.no_grad():
        for _ in range(N_WARM):
            with autocast("cuda", enabled=cfg.AMP):
                mdl(_vid_p, _ids_p, _amask_p, _ttids_p, training=True)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = _time_prof.perf_counter()
        for _ in range(N_RUN):
            with autocast("cuda", enabled=cfg.AMP):
                mdl(_vid_p, _ids_p, _amask_p, _ttids_p, training=True)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        res["latency_ms"] = round((_time_prof.perf_counter() - t0) / N_RUN * 1000, 2)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(0)
        with torch.no_grad():
            with autocast("cuda", enabled=cfg.AMP):
                mdl(_vid_p, _ids_p, _amask_p, _ttids_p, training=True)
        res["vram_peak_MB"] = round(torch.cuda.max_memory_allocated(0) / 1024**2, 1)
    else:
        res["vram_peak_MB"] = None

    with torch.no_grad():
        with autocast("cuda", enabled=cfg.AMP):
            fv, _, _ = mdl.encode(_vid_p, _ids_p, _amask_p, _ttids_p)
    res["video_tokens"] = fv.shape[1]

    return res


r = profile_model("ViT", model)
PROFILE_RESULTS["ViT"] = r
print(f"\n{'='*55}\n  MODEL: ViT\n{'='*55}")
print(f"  Total params           : {r['params_total_M']:.2f} M")
print(f"  Trainable params       : {r['params_trainable_M']:.2f} M")
print(f"  Video encoder params   : {r['params_video_enc_M']:.2f} M")
gfl = r['gflops_video_enc']
if gfl is not None: print(f"  GFLOPs (video encoder) : {gfl:.2f}")
print(f"  Inference latency      : {r['latency_ms']:.2f} ms / sample")
if r['vram_peak_MB']: print(f"  Peak VRAM (fwd pass)   : {r['vram_peak_MB']:.1f} MB")
print(f"  Video tokens (Lv)      : {r['video_tokens']}")
print("\n[profiler] Done. Results in PROFILE_RESULTS['ViT']")


## Cell 21 — Full Training Execution

In [ ]:
# # Cell 21: Full Training Execution — ViT (with Early Stopping)
# # ============================================================

# train_loader, val_loader, test_loader = build_dataloaders(cfg, tokenizer)

# total_steps = max(len(train_loader) // cfg.GRAD_ACCUM, 1) * cfg.NUM_EPOCHS
# optimizer, scheduler, scaler = build_optimizer_and_scheduler(model, cfg, total_steps)
# writer      = SummaryWriter(log_dir=str(cfg.LOG_DIR))
# early_stop  = EarlyStopping(
#     patience     = cfg.ES_PATIENCE,
#     min_delta    = cfg.ES_MIN_DELTA,
#     mode         = cfg.ES_MODE,
#     restore_best = True,
#     verbose      = True,
# )

# start_epoch  = 0
# best_metric  = 0.0
# global_step  = 0
# epoch        = -1
# latest_ckpt  = cfg.CKPT_DIR / "latest.pt"

# if latest_ckpt.exists():
#     start_epoch = load_checkpoint(
#         latest_ckpt, model, optimizer, scheduler, scaler, DEVICE
#     ) + 1

# if start_epoch >= cfg.NUM_EPOCHS:
#     logger.warning(
#         f"[ViT] Checkpoint is at epoch {start_epoch} but NUM_EPOCHS={cfg.NUM_EPOCHS}. "
#         f"Nothing to train. Increase cfg.NUM_EPOCHS or delete the checkpoint."
#     )
# else:
#     logger.info(
#         f"[ViT] Training epoch {start_epoch+1} → {cfg.NUM_EPOCHS}  "
#         f"(early stopping: patience={cfg.ES_PATIENCE}, "
#         f"min_delta={cfg.ES_MIN_DELTA}%)"
#     )

#     for epoch in range(start_epoch, cfg.NUM_EPOCHS):
#         model.train()
#         avg_loss, global_step = train_one_epoch(
#             model, train_loader, optimizer, scheduler, scaler,
#             criterion, cfg, DEVICE, epoch, writer, global_step,
#         )

#         val_metrics = evaluate(model, val_loader, cfg, DEVICE)
#         primary     = val_metrics.get("R@1_IoU0.5", 0.0)

#         logger.info(
#             f"[ViT] Epoch {epoch+1:>3d}/{cfg.NUM_EPOCHS} | "
#             f"loss {avg_loss:.4f} | "
#             + " | ".join(f"{k}: {v:.2f}%" for k, v in val_metrics.items())
#         )

#         for k, v in val_metrics.items():
#             writer.add_scalar(f"vit/val/{k}", v, epoch)

#         if primary > best_metric:
#             best_metric = primary
#             save_checkpoint(
#                 model, optimizer, scheduler, scaler, epoch, val_metrics,
#                 cfg.CKPT_DIR / "best.pt",
#             )

#         save_checkpoint(model, optimizer, scheduler, scaler, epoch, {}, latest_ckpt)

#         if early_stop(primary, model, epoch):
#             logger.info(
#                 f"[ViT] Early stopping triggered at epoch {epoch+1}. "
#                 f"Best R@1 IoU@0.5 = {early_stop.best_score:.2f}%"
#             )
#             save_checkpoint(
#                 model, optimizer, scheduler, scaler, epoch, val_metrics,
#                 cfg.CKPT_DIR / "best_es.pt",
#             )
#             break

#     writer.close()
#     stopped_at = (early_stop.stopped_epoch if early_stop.triggered else epoch)
#     logger.info(
#         f"[ViT] Training complete. "
#         f"Best R@1 IoU@0.5 = {early_stop.best_score or best_metric:.2f}%  "
#         f"(stopped at epoch {stopped_at + 1}/{cfg.NUM_EPOCHS})"
#     )


# Cell 21b: Train Both Models with Early Stopping — Comparison Metrics


In [ ]:
# Cell 21b: Train Both Models with Early Stopping — Comparison Metrics
# ============================================================
# RESUME LOGIC: Each backbone saves <ckpt_dir>/<bname>/latest.pt
# hist is checkpointed so COMPARISON_RESULTS is fully restored on resume.
# ============================================================
import time as _time_dt

train_loader, val_loader, test_loader = build_dataloaders(cfg, tokenizer)
total_steps = max(len(train_loader) // cfg.GRAD_ACCUM, 1) * cfg.TOTAL_EPOCHS

COMPARISON_RESULTS = {}


@torch.no_grad()
def evaluate_subset(model, loader, cfg, device, max_batches=30):
    model.eval()
    hits = 0; total = 0
    for i, batch in enumerate(loader):
        if i >= max_batches:
            break
        video          = batch["video"].to(device)
        input_ids      = batch["input_ids"].to(device, dtype=torch.long)
        attention_mask = batch["attention_mask"].to(device, dtype=torch.long)
        token_type_ids = batch["token_type_ids"].to(device, dtype=torch.long)
        start_gt       = batch["start_norm"]
        end_gt         = batch["end_norm"]
        span_mask      = batch["span_mask"]

        amp_device = "cuda" if device.type == "cuda" else "cpu"
        with autocast(amp_device, enabled=cfg.AMP and device.type == "cuda"):
            out = model(video, input_ids, attention_mask, token_type_ids, training=True)

        start_preds = out["start_preds"].cpu()
        end_preds   = out["end_preds"].cpu()
        conf_probs  = torch.sigmoid(out["conf_logits"]).cpu()

        for b in range(video.shape[0]):
            best_idx  = conf_probs[b].argmax().item()
            ps, pe    = start_preds[b, best_idx].item(), end_preds[b, best_idx].item()
            valid_idx = span_mask[b].nonzero(as_tuple=True)[0]
            if not len(valid_idx):
                continue
            fi       = valid_idx[0].item()
            gs, ge   = start_gt[b, fi].item(), end_gt[b, fi].item()
            if temporal_iou(ps, pe, gs, ge) >= 0.5:
                hits += 1
            total += 1

    model.train()
    return (hits / max(total, 1)) * 100.0


def save_experiment_checkpoint(mdl, opt, sched, scaler, es, epoch,
                                hist, best_r05, gstep, val_m, path):
    save_checkpoint(mdl, opt, sched, scaler, epoch, val_m, path)
    ckpt = torch.load(path, map_location="cpu")
    ckpt.update({
        "hist":     hist,
        "best_r05": best_r05,
        "gstep":    gstep,
        "es_state": {
            "counter":       es.counter,
            "best_score":    es.best_score,
            "triggered":     es.triggered,
            "stopped_epoch": getattr(es, "stopped_epoch", -1),
        },
    })
    torch.save(ckpt, path)


def load_experiment_checkpoint(path, mdl, opt, sched, scaler, es, device):
    epoch_done = load_checkpoint(path, mdl, opt, sched, scaler, device)
    ckpt       = torch.load(path, map_location="cpu")

    hist     = ckpt.get("hist",     {})
    best_r05 = ckpt.get("best_r05", 0.0)
    gstep    = ckpt.get("gstep",    0)

    es_state = ckpt.get("es_state", {})
    if es_state:
        es.counter    = es_state.get("counter",    0)
        es.best_score = es_state.get("best_score", None)
        es.triggered  = es_state.get("triggered",  False)
        if hasattr(es, "stopped_epoch"):
            es.stopped_epoch = es_state.get("stopped_epoch", -1)

    return epoch_done + 1, hist, best_r05, gstep


def run_experiment(bname: str, mdl: nn.Module) -> dict:
    print(f"\n{'='*60}\n  EXPERIMENT: {bname}\n{'='*60}")

    crit               = TemporalGroundingLossV2(lambda_l1=5.0, lambda_iou=2.0, lambda_conf=1.0).to(DEVICE)
    opt, sched, scaler = build_optimizer_and_scheduler(mdl, cfg, total_steps)
    tb                 = SummaryWriter(log_dir=str(cfg.LOG_DIR / bname))
    ckpt_dir           = cfg.CKPT_DIR / bname
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    latest_ckpt        = ckpt_dir / "latest.pt"

    es = EarlyStopping(
        patience     = cfg.ES_PATIENCE,
        min_delta    = cfg.ES_MIN_DELTA,
        mode         = cfg.ES_MODE,
        restore_best = True,
        verbose      = True,
    )

    hist, best_r05, gstep = {}, 0.0, 0
    start_epoch = 0
    epoch       = -1

    if latest_ckpt.exists():
        start_epoch, hist, best_r05, gstep = load_experiment_checkpoint(
            latest_ckpt, mdl, opt, sched, scaler, es, DEVICE
        )
        logger.info(
            f"[{bname}] Resumed from {latest_ckpt}. "
            f"Continuing epoch {start_epoch+1} → {cfg.NUM_EPOCHS} | "
            f"ES counter={es.counter}/{cfg.ES_PATIENCE} | best_r05={best_r05:.2f}%"
        )
        # FIX: reset dangerously low AMP scaler after resume
        if cfg.AMP and hasattr(scaler, "_scale") and scaler._scale is not None:
            current_scale = scaler.get_scale()
            logger.info(f"[{bname}] AMP scaler restored: scale={current_scale:.1f}")
            if current_scale < 1.0:
                logger.warning(
                    f"[{bname}] AMP scaler scale={current_scale} too low. "
                    f"Resetting to 2**16."
                )
                scaler._scale = torch.tensor(2.0 ** 16, dtype=torch.float32)
    else:
        logger.info(f"[{bname}] No checkpoint — starting from scratch. Epochs 1 → {cfg.NUM_EPOCHS}")

    if start_epoch >= cfg.NUM_EPOCHS:
        logger.warning(
            f"[{bname}] Already at epoch {start_epoch}/{cfg.NUM_EPOCHS}. "
            f"Delete {latest_ckpt} or increase cfg.NUM_EPOCHS."
        )
        tb.close()
        return hist

    if es.triggered:
        logger.warning(f"[{bname}] Early stopping already triggered. Best={es.best_score:.2f}%")
        tb.close()
        return hist

    for epoch in range(start_epoch, cfg.NUM_EPOCHS):
        t0 = _time_dt.perf_counter()
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats(0)

        avg_loss, gstep = train_one_epoch(
            mdl, train_loader, opt, sched, scaler, crit, cfg, DEVICE,
            epoch, tb, gstep,
        )

        epoch_secs = _time_dt.perf_counter() - t0
        peak_vram  = (
            torch.cuda.max_memory_allocated(0) / 1024**2
            if DEVICE.type == "cuda" else 0.0
        )

        train_acc = evaluate_subset(mdl, train_loader, cfg, DEVICE, max_batches=10)
        val_m     = evaluate(mdl, val_loader, cfg, DEVICE)
        val_acc   = val_m.get("R@1_IoU0.5", 0.0)

        hist[epoch] = {
            "train_loss":    avg_loss,
            "train_acc":     round(train_acc, 2),
            "val_acc":       round(val_acc, 2),
            "epoch_secs":    round(epoch_secs, 2),
            "peak_vram_MB":  round(peak_vram, 1),
            "es_counter":    es.counter,
            "es_best_score": es.best_score or 0.0,
            **val_m,
        }

        for k, v in val_m.items():
            tb.add_scalar(f"{bname}/val/{k}", v, epoch)
        tb.add_scalar(f"{bname}/train/acc_r1_iou05", train_acc, epoch)
        tb.add_scalar(f"{bname}/val/acc_r1_iou05",   val_acc,   epoch)
        tb.add_scalar(f"{bname}/es/counter",          es.counter, epoch)

        primary = val_acc

        if primary > best_r05:
            best_r05 = primary
            save_experiment_checkpoint(
                mdl, opt, sched, scaler, es, epoch,
                hist, best_r05, gstep, val_m, ckpt_dir / "best.pt",
            )

        save_experiment_checkpoint(
            mdl, opt, sched, scaler, es, epoch,
            hist, best_r05, gstep, val_m, latest_ckpt,
        )

        logger.info(
            f"[{bname}] Ep {epoch+1:>2}/{cfg.NUM_EPOCHS} | "
            f"loss={avg_loss:.4f} | "
            f"train_acc={train_acc:.1f}% | val_acc={val_acc:.1f}% "
            f"(R@1 IoU@0.3={val_m.get('R@1_IoU0.3',0):.1f}% "
            f"IoU@0.7={val_m.get('R@1_IoU0.7',0):.1f}%) | "
            f"{epoch_secs:.0f}s | {peak_vram:.0f} MB | "
            f"ES {es.counter}/{cfg.ES_PATIENCE}"
        )

        if es(primary, mdl, epoch):
            logger.info(
                f"[{bname}] Early stopping at epoch {epoch+1}. "
                f"Best val_acc={es.best_score:.2f}%"
            )
            save_experiment_checkpoint(
                mdl, opt, sched, scaler, es, epoch,
                hist, best_r05, gstep, val_m, ckpt_dir / "best_es.pt",
            )
            break

    tb.close()
    stopped_at = (es.stopped_epoch if es.triggered else epoch)
    logger.info(
        f"[{bname}] Experiment complete. "
        f"Best val_acc={es.best_score or best_r05:.2f}% "
        f"(stopped at epoch {stopped_at + 1}/{cfg.NUM_EPOCHS})"
    )
    return hist


# Run ViT training experiment
COMPARISON_RESULTS["ViT"] = run_experiment("ViT", model)

logger.info("[ViT-train] Experiment complete. Run Cell 21c for results.")

# Cell 21c: Swin vs ViT -- Side-by-Side Comparison Plots


In [ ]:
# Cell 21c: ViT Training Results — Comprehensive Plots
# ============================================================
# Visualises training/validation accuracy and loss over epochs.
# Requires COMPARISON_RESULTS from Cell 21b.
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

VIT_C = '#FF5722'

vit_hist = COMPARISON_RESULTS.get("ViT", {})
epochs   = sorted(vit_hist.keys())
eps1     = [e + 1 for e in epochs]

def extract(hist, key):
    return [hist.get(e, {}).get(key, float('nan')) for e in epochs]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Video ViT — Training Results on CharadesSTA",
             fontsize=14, fontweight='bold')

def plot(ax, key, title, ylabel):
    ax.plot(eps1, extract(vit_hist, key), '-o', color=VIT_C, lw=2, ms=4)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Epoch', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(alpha=0.25)

plot(axes[0,0], 'train_loss',  'Training Loss',            'Loss')
plot(axes[0,1], 'train_acc',   'Train Acc R@1 IoU@0.5 (%)', 'Accuracy (%)')
plot(axes[0,2], 'val_acc',     'Val Acc R@1 IoU@0.5 (%)',   'Accuracy (%)')
plot(axes[1,0], 'R@1_IoU0.3', 'Val R@1 IoU@0.3 (%)',       'Accuracy (%)')
plot(axes[1,1], 'R@1_IoU0.5', 'Val R@1 IoU@0.5 (%)',       'Accuracy (%)')
plot(axes[1,2], 'R@1_IoU0.7', 'Val R@1 IoU@0.7 (%)',       'Accuracy (%)')

plt.tight_layout()
plt.savefig('vit_training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Saved: vit_training_results.png')

# ── Summary table ──────────────────────────────────────────────
def last_valid(hist, key):
    for e in sorted(hist.keys(), reverse=True):
        v = hist.get(e, {}).get(key, float('nan'))
        if v == v and v > 0:
            return round(v, 2)
    return 0.0

print('\n' + '='*50)
print('  ViT — BEST VALIDATION METRICS')
print('='*50)
for key in ['R@1_IoU0.3', 'R@1_IoU0.5', 'R@1_IoU0.7']:
    print(f'  {key:<22}: {last_valid(vit_hist, key):.2f}%')
print('='*50)


In [ ]:
from pathlib import Path

BASE     = Path(".")
ANNO_DIR = BASE / "CHARADES_STA_DATASET" / "annotations" / "annotations"/ "annotations"
TRAIN_TXT = ANNO_DIR / "charades_sta_train.txt"
TEST_TXT  = ANNO_DIR / "charades_sta_test.txt"

print("ANNO_DIR  :", ANNO_DIR,  "→ exists:", ANNO_DIR.exists())
print("TRAIN_TXT :", TRAIN_TXT, "→ exists:", TRAIN_TXT.exists())
print("TEST_TXT  :", TEST_TXT,  "→ exists:", TEST_TXT.exists())

# Check file sizes
for p in [TRAIN_TXT, TEST_TXT]:
    if p.exists():
        size = p.stat().st_size
        print(f"  {p.name} → {size} bytes")
        # Show first 2 lines
        with open(p, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 2: break
                print(f"  Line {i+1}: {line.strip()}")

## Cell 22 — Test Set Evaluation

In [ ]:
# Cell 22: Test Set Evaluation — ViT
# ============================================================

TEST_RESULTS = {}

best_ckpt = cfg.CKPT_DIR / "ViT" / "best.pt"
if not best_ckpt.exists():
    best_ckpt = cfg.CKPT_DIR / "best.pt"

if best_ckpt.exists():
    load_checkpoint(best_ckpt, model, device=DEVICE)
    test_m = evaluate(model, test_loader, cfg, DEVICE)
    TEST_RESULTS["ViT"] = test_m

    logger.info("=" * 60)
    logger.info("[ViT] TEST SET RESULTS")
    for k, v in test_m.items():
        logger.info(f"  {k:<20s}: {v:.2f}%")
    logger.info("=" * 60)
else:
    logger.warning(
        f"[ViT] No best checkpoint found at {best_ckpt}. "
        f"Run Cell 21b first."
    )
    TEST_RESULTS["ViT"] = {}


In [ ]:
cfg.VIDEO_DIR = Path("./CHARADES_STA_DATASET/videos/videos/videos")
print("Updated cfg.VIDEO_DIR:", cfg.VIDEO_DIR, "| exists:", cfg.VIDEO_DIR.exists())

## Cell 23 — Inference: Single Video + Text Query

In [ ]:
# Cell 23: Inference — Single Video + Natural Language Query
# ============================================================

def predict(video_path, query, model, tokenizer, cfg, device):
    model.eval()
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    dur   = total / max(fps, 1e-6)

    indices = np.linspace(0, max(total-1,0), cfg.NUM_FRAMES, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: frame = frames[-1].copy() if frames else np.zeros((cfg.FRAME_H, cfg.FRAME_W, 3), dtype=np.uint8)
        frame = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (cfg.FRAME_W, cfg.FRAME_H))
        frames.append(frame)
    cap.release()

    MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1,1)
    STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1,1)
    vt   = torch.from_numpy(np.stack(frames,0).astype(np.float32)/255.0).permute(3,0,1,2)
    vt   = ((vt - MEAN) / STD).unsqueeze(0).to(device)

    enc = tokenizer(query, max_length=cfg.MAX_TEXT_LEN, padding="max_length",
                    truncation=True, return_tensors="pt")
    with torch.no_grad(), autocast("cuda", enabled=cfg.AMP):
        result = model(vt, enc["input_ids"].to(device),
                       enc["attention_mask"].to(device),
                       enc["token_type_ids"].to(device),
                       duration=torch.tensor([dur]).to(device), training=False)
    return result


DEMO_VIDEO = "./CHARADES_STA_DATASET/videos/videos/videos/00N38.mp4"
DEMO_QUERY = "The boxer in white trunks lands a series of punches on his opponent."

if Path(DEMO_VIDEO).exists():
    out = predict(DEMO_VIDEO, DEMO_QUERY, model, tokenizer, cfg, DEVICE)
    # BUG 2 FIX: forward(training=False) returns {'spans': [...]} only.
    # The 'message' key was removed in v5 (top-1 fallback guarantees non-empty
    # spans, so the 'no activity found' sentinel is never reachable).
    # Accessing out['message'] here was a guaranteed KeyError.
    pred_spans = out['spans'][0]   # list of (start_sec, end_sec) tuples
    print(f"Query : '{DEMO_QUERY}'")
    if pred_spans:
        for i, (s, e) in enumerate(pred_spans):
            print(f"  Span {i+1}: {s:.2f}s → {e:.2f}s")
    else:
        print("  (no spans above confidence threshold — try lowering cfg.CONF_THRESHOLD)")
else:
    logger.info("Demo video not found. Place CharadesSTA dataset first.")

## Cell 24 — Visualise Predictions

In [ ]:
# Cell 24: Timeline Visualisation (unchanged from v1)
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def visualise_spans(pred_spans, gt_spans, duration, query):
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.set_xlim(0, duration); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel("Time (seconds)", fontsize=12)
    ax.set_title(f'[v2] Query: "{query}"', fontsize=11, pad=10)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Predicted", "Ground Truth"])
    for s, e in gt_spans:
        ax.barh(1, e-s, left=s, height=0.4, color="#2ecc71", alpha=0.8)
    for s, e in pred_spans:
        ax.barh(0, e-s, left=s, height=0.4, color="#3498db", alpha=0.8)
        best_iou = max((temporal_iou(s,e,gs,ge) for gs,ge in gt_spans), default=0.0)
        ax.text((s+e)/2, 0, f"IoU={best_iou:.2f}",
                ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    ax.legend(handles=[mpatches.Patch(color="#3498db", label="v2 Prediction"),
                       mpatches.Patch(color="#2ecc71", label="Ground Truth")], loc="upper right")
    plt.tight_layout()
    plt.savefig("prediction_vis_v2.png", dpi=150, bbox_inches="tight")
    plt.show()


DEMO_GT  = [(0.0, 2.5)]
DEMO_DUR = 8.0
try:
    visualise_spans(out["spans"][0] if out["spans"] else [], DEMO_GT, DEMO_DUR, DEMO_QUERY)
except NameError:
    logger.info("Run Cell 23 first.")

## Cell 25 — Fixes Summary & GPU Memory Report

In [ ]:
# Cell 25: Fixes Summary & GPU Memory Report
# ============================================================

if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated(0) / 1e9
    reserv = torch.cuda.memory_reserved(0) / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"=== GPU Memory Report — RTX 4080 Super ===")
    print(f"  Allocated : {alloc:.2f} GB")
    print(f"  Reserved  : {reserv:.2f} GB")
    print(f"  Free      : {total - reserv:.2f} GB")
    print(f"  Total     : {total:.1f} GB")
    try:
        stats = torch.cuda.memory_stats(0)
        peak  = stats.get("allocated_bytes.all.peak", 0) / 1e9
        print(f"  Peak alloc: {peak:.2f} GB  (since last torch.cuda.reset_peak_memory_stats())")
    except Exception:
        pass

print("""
══════════════════════════════════════════════════════════════════════
 v2 ARCHITECTURE CHANGES vs v1 — COMPLETE SUMMARY
══════════════════════════════════════════════════════════════════════

 🔴 FIX 1 — Causal mask on encoder (Cell 12)
   v1: make_causal_mask() applied to cross-modal encoder
       → video token t5 CANNOT attend to t6 (wrong)
   v2: NO causal mask anywhere in encoder
       → full bidirectional attention as intended

 🔴 FIX 2 — GAP destroys temporal structure (Cell 13)
   v1: video_desc = GAP(fused_video)  (B, Lv, D) → (B, D)
       → all temporal position info lost before span prediction
   v2: fused_video (B, Lv, D) passed directly to span head
       → each query attends selectively to different time regions

 🔴 FIX 3 — Weak cross-modal fusion (Cell 12)
   v1: joint = cat([video, text])  →  self_attn(joint)
       → model must implicitly discover video/text boundaries
   v2: BidirectionalCrossModalFusion:
       video ← CrossAttn(Q=video, K=text, V=text)
       text  ← CrossAttn(Q=text, K=video, V=video)
       Both × NUM_FUSION_LAYERS=4 with within-modality self-attn

 🟡 FIX 4 — No modality distinction in PE (Cell 11)
   v1: sinusoidal PE on joint sequence — position 84 could
       be video or text with no signal which
   v2: ModalityTypeEmbedding (learned, like BERT segment embeds)
       video tokens += embed_video;  text tokens += embed_text
       applied BEFORE PE — explicit modality identity from step 1

 🟡 FIX 5 — Projection too shallow (Cell 10)
   v1: 2-layer MLP: 768 → 256 → 256  (67% collapse in 1 step)
   v2: ResidualProjection: 768 → 640 → 512  + linear skip 768→512
       Residual prevents gradient vanishing through bottleneck
       SHARED_DIM: 256 → 512

 🟡 FIX 6 — LSTM teacher-forcing mismatch (Cell 13)
   v1: LSTM autoregressively predicts spans (train=teacher-forced,
       inference=free-run → exposure bias, error accumulation)
   v2: DETR-style LearnedQuerySpanHead:
       K learnable queries attend to all Lv video tokens in parallel
       All K spans predicted in one pass — zero teacher-forcing gap
       Hungarian matching assigns predictions to GT spans for loss

 🟢 FIX 7 — Training from scratch on tiny dataset
   VideoViT random init on CharadesSTA (~10K videos).
   For improved accuracy, a pretrained ViT checkpoint (e.g.
   MAE or DINO pretrained on ImageNet) can be loaded via
   cfg.PRETRAINED_VIT_PATH and adapted for 3D video input.
   Expected gain: +3-6% R@1 IoU@0.5 vs random init.

══════════════════════════════════════════════════════════════════════
 TensorBoard: tensorboard --logdir=./logs_v2 --port=6006
══════════════════════════════════════════════════════════════════════
""")

# Cell 26 — Computational Complexity Visualisation


In [ ]:
# Cell 26: ViT Inference Complexity Visualisation
# ============================================================
# Uses PROFILE_RESULTS['ViT'] from Cell 20b.
# Plots: latency, peak VRAM, parameter count, token count.
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

pr = PROFILE_RESULTS.get('ViT', {})

metrics = {
    'Total Params (M)':     pr.get('params_total_M',     0),
    'Video Enc Params (M)': pr.get('params_video_enc_M', 0),
    'GFLOPs (video enc)':   pr.get('gflops_video_enc')  or 0,
    'Latency (ms)':         pr.get('latency_ms',         0),
    'Peak VRAM (MB)':       pr.get('vram_peak_MB')       or 0,
    'Video tokens (Lv)':    pr.get('video_tokens',       0),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Video ViT — Complexity Profile', fontsize=13, fontweight='bold')

VIT_C = '#FF5722'

# Left: parameter & token counts
ax1 = axes[0]
keys1 = ['Total Params (M)', 'Video Enc Params (M)', 'Video tokens (Lv)']
vals1 = [metrics[k] for k in keys1]
bars1 = ax1.bar(keys1, vals1, color=VIT_C, alpha=0.85, edgecolor='white')
ax1.set_title('Parameters & Tokens', fontweight='bold')
ax1.set_ylabel('Value')
ax1.grid(axis='y', alpha=0.25)
for bar, v in zip(bars1, vals1):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02,
             f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

# Right: compute & memory metrics
ax2 = axes[1]
keys2 = ['GFLOPs (video enc)', 'Latency (ms)', 'Peak VRAM (MB)']
vals2 = [metrics[k] for k in keys2]
bars2 = ax2.bar(keys2, vals2, color=VIT_C, alpha=0.85, edgecolor='white')
ax2.set_title('Compute & Memory', fontweight='bold')
ax2.set_ylabel('Value (mixed units)')
ax2.grid(axis='y', alpha=0.25)
for bar, v in zip(bars2, vals2):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02,
             f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('vit_complexity_profile.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*50)
print('  ViT COMPLEXITY PROFILE')
print('='*50)
for k, v in metrics.items():
    print(f'  {k:<28}: {v}')
fps = round(1000 / max(metrics['Latency (ms)'], 1e-3), 2)
print(f'  {"FPS (inferred)":<28}: {fps}')
print('='*50)
print('[OK] Saved: vit_complexity_profile.png')
